# Business Impact & Adaptive Portfolio Decision System

## Objective

Translate the validated NIFTY 50 volatility forecasting research into a
business-oriented market-risk decision framework.

The notebook consumes only the frozen Research Output v1 files and does not
depend on variables stored in the research notebook's kernel.

## Business Question

How can an investment-management firm adjust NIFTY 50 market exposure
when expected market volatility changes?

## Research Input

Research Output Version: v1

Source:
`data/research_outputs/v1/`

## Methodological Principle

Research calculations remain frozen.

This notebook performs downstream business analysis only.

In [1]:
# ============================================================
# LOAD FROZEN RESEARCH OUTPUT v1
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"D:\DS Projects\Adaptive Market Risk Intelligence System"
)


# ------------------------------------------------------------
# 2. AUTHORITATIVE RESEARCH OUTPUT LOCATION
# ------------------------------------------------------------

RESEARCH_OUTPUT_VERSION = "v1"

RESEARCH_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "research_outputs"
    / RESEARCH_OUTPUT_VERSION
)


# ------------------------------------------------------------
# 3. REQUIRED FILES
# ------------------------------------------------------------

OOS_FILE = (
    RESEARCH_OUTPUT_DIR
    / "oos_forecasts.csv"
)

PERFORMANCE_FILE = (
    RESEARCH_OUTPUT_DIR
    / "model_performance.csv"
)

METADATA_FILE = (
    RESEARCH_OUTPUT_DIR
    / "research_metadata.json"
)

MANIFEST_FILE = (
    RESEARCH_OUTPUT_DIR
    / "output_manifest.json"
)

# ------------------------------------------------------------
# 4. BUSINESS ANALYSIS CONFIGURATION
# ------------------------------------------------------------

BUSINESS_DEVELOPMENT_END = pd.Timestamp("2025-12-31")
BUSINESS_EVALUATION_START = pd.Timestamp("2026-01-01")

BUSINESS_EXPECTED_DEVELOPMENT_N = 495
BUSINESS_EXPECTED_EVALUATION_N = 178

BUSINESS_EXPECTED_OOS_START = pd.Timestamp("2024-01-01")
BUSINESS_EXPECTED_OOS_END = pd.Timestamp("2026-09-22")

# ------------------------------------------------------------
# 5. CHECK FILE EXISTENCE
# ------------------------------------------------------------

required_files = [
    OOS_FILE,
    PERFORMANCE_FILE,
    METADATA_FILE,
    MANIFEST_FILE,
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required Research Output v1 files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )


# ------------------------------------------------------------
# 6. LOAD AUTHORITATIVE DATA
# ------------------------------------------------------------

biz_oos = pd.read_csv(
    OOS_FILE,
    parse_dates=["Date"]
)

biz_model_performance = pd.read_csv(
    PERFORMANCE_FILE,
    parse_dates=["OOS_Start", "OOS_End"]
)

with open(
    METADATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    biz_research_metadata = json.load(f)


# ------------------------------------------------------------
# 7. BASIC LOAD REPORT
# ------------------------------------------------------------

print("=" * 80)
print("RESEARCH OUTPUT v1 LOADED")
print("=" * 80)

print("Research version :", RESEARCH_OUTPUT_VERSION)
print("OOS observations :", len(biz_oos))
print(
    "OOS date range   :",
    biz_oos["Date"].min().date(),
    "to",
    biz_oos["Date"].max().date()
)

print("Performance rows :", len(biz_model_performance))
print("Metadata loaded  :", isinstance(biz_research_metadata, dict))

print()
print("OOS file:")
print(OOS_FILE)

RESEARCH OUTPUT v1 LOADED
Research version : v1
OOS observations : 673
OOS date range   : 2024-01-01 to 2026-09-22
Performance rows : 8
Metadata loaded  : True

OOS file:
D:\DS Projects\Adaptive Market Risk Intelligence System\data\research_outputs\v1\oos_forecasts.csv


In [2]:
# ============================================================
# VALIDATE IMPORTED RESEARCH OUTPUT v1
# ============================================================

BUSINESS_EXPECTED_COLUMNS = [
    "Date",
    "NIFTY_Log_Return",
    "Actual_Volatility",
    "India_VIX_PreReturn",
    "Historical_20D",
    "EWMA",
    "GARCH_t",
    "GJR_GARCH_t",
    "EGARCH_t",
    "Historical_20D_VIX",
    "EWMA_VIX",
    "EGARCH_t_VIX",
]

BUSINESS_EXPECTED_N = 673
BUSINESS_EXPECTED_START = pd.Timestamp("2024-01-01")
BUSINESS_EXPECTED_END = pd.Timestamp("2026-09-22")


# ------------------------------------------------------------
# 1. COLUMN VALIDATION
# ------------------------------------------------------------

missing_columns = [
    col
    for col in BUSINESS_EXPECTED_COLUMNS
    if col not in biz_oos.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Research v1 columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 2. OBSERVATION COUNT
# ------------------------------------------------------------

if len(biz_oos) != BUSINESS_EXPECTED_N:
    raise ValueError(
        f"Expected {BUSINESS_EXPECTED_N} observations, "
        f"found {len(biz_oos)}."
    )


# ------------------------------------------------------------
# 3. DATE VALIDATION
# ------------------------------------------------------------

if biz_oos["Date"].min() != BUSINESS_EXPECTED_START:
    raise ValueError("BUSINESS_EXPECTED_OOS_START")

if biz_oos["Date"].max() != BUSINESS_EXPECTED_END:
    raise ValueError("BUSINESS_EXPECTED_OOS_END")


# ------------------------------------------------------------
# 4. DUPLICATE DATE VALIDATION
# ------------------------------------------------------------

duplicate_dates = (
    biz_oos["Date"]
    .duplicated()
    .sum()
)

if duplicate_dates != 0:
    raise ValueError(
        f"Duplicate dates detected: {duplicate_dates}"
    )


# ------------------------------------------------------------
# 5. MISSING VALUE VALIDATION
# ------------------------------------------------------------

missing_counts = (
    biz_oos[BUSINESS_EXPECTED_COLUMNS]
    .isna()
    .sum()
)

if missing_counts.sum() != 0:
    raise ValueError(
        "Missing values detected:\n"
        + missing_counts[
            missing_counts > 0
        ].to_string()
    )


# ------------------------------------------------------------
# 6. NON-FINITE VALUE VALIDATION
# ------------------------------------------------------------

numeric_columns = [
    col
    for col in BUSINESS_EXPECTED_COLUMNS
    if col != "Date"
]

if not np.isfinite(
    biz_oos[numeric_columns].to_numpy()
).all():

    raise ValueError(
        "Infinite or non-finite values detected."
    )


# ------------------------------------------------------------
# 7. CHRONOLOGICAL VALIDATION
# ------------------------------------------------------------

if not biz_oos["Date"].is_monotonic_increasing:
    raise ValueError(
        "Dates are not chronologically ordered."
    )


# ------------------------------------------------------------
# 8. RESEARCH VERSION VALIDATION
# ------------------------------------------------------------

if biz_research_metadata.get(
    "research_version"
) != RESEARCH_OUTPUT_VERSION:

    raise ValueError(
        "Research metadata version does not match "
        "the loaded Research Output version."
    )


# ------------------------------------------------------------
# 9. FINAL VALIDATION REPORT
# ------------------------------------------------------------

print("=" * 80)
print("BUSINESS INPUT VALIDATION: PASSED")
print("=" * 80)

print("Research version :", RESEARCH_OUTPUT_VERSION)
print("Observations     :", len(biz_oos))
print(
    "Date range       :",
    biz_oos["Date"].min().date(),
    "to",
    biz_oos["Date"].max().date()
)
print("Missing values   :", 0)
print("Duplicate dates  :", duplicate_dates)
print("Non-finite       :", 0)
print("Columns          :", len(BUSINESS_EXPECTED_COLUMNS))
print("Status           : VALID")

BUSINESS INPUT VALIDATION: PASSED
Research version : v1
Observations     : 673
Date range       : 2024-01-01 to 2026-09-22
Missing values   : 0
Duplicate dates  : 0
Non-finite       : 0
Columns          : 12
Status           : VALID


In [3]:
# ============================================================
# SELECT PRIMARY BUSINESS RISK SIGNAL
# ============================================================

# ------------------------------------------------------------
# 1. Restrict model selection to business-development period
# ------------------------------------------------------------

biz_model_selection_data = (
    biz_oos[
        biz_oos["Date"] <= BUSINESS_DEVELOPMENT_END
    ]
    .copy()
)

if biz_model_selection_data.empty:
    raise ValueError(
        "No observations available for business model selection."
    )


# ------------------------------------------------------------
# 2. Model-to-signal mapping
# ------------------------------------------------------------

BUSINESS_SIGNAL_MAP = {
    "Historical_20D": "Historical_20D",
    "EWMA": "EWMA",
    "GARCH_t": "GARCH_t",
    "GJR_GARCH_t": "GJR_GARCH_t",
    "EGARCH_t": "EGARCH_t",
    "Historical_20D_VIX": "Historical_20D_VIX",
    "EWMA_VIX": "EWMA_VIX",
    "EGARCH_t_VIX": "EGARCH_t_VIX",
}


# ------------------------------------------------------------
# 3. Calculate development-period model performance
# ------------------------------------------------------------

biz_model_selection_results = []

for biz_model, biz_signal_column in BUSINESS_SIGNAL_MAP.items():

    if biz_signal_column not in biz_model_selection_data.columns:
        raise ValueError(
            f"Required signal column missing: {biz_signal_column}"
        )

    biz_valid = (
        biz_model_selection_data[
            [
                "Actual_Volatility",
                biz_signal_column
            ]
        ]
        .dropna()
        .copy()
    )

    if biz_valid.empty:
        raise ValueError(
            f"No valid observations available for {biz_model}."
        )

    biz_errors = (
        biz_valid[biz_signal_column]
        - biz_valid["Actual_Volatility"]
    )

    biz_mae = np.mean(
        np.abs(biz_errors)
    )

    biz_rmse = np.sqrt(
        np.mean(biz_errors ** 2)
    )

    biz_model_selection_results.append(
        {
            "Model": biz_model,
            "MAE": biz_mae,
            "RMSE": biz_rmse,
            "N": len(biz_valid)
        }
    )


# ------------------------------------------------------------
# 4. Rank models using development-period RMSE
# ------------------------------------------------------------

biz_model_ranking = (
    pd.DataFrame(biz_model_selection_results)
    .sort_values(
        by=["RMSE", "MAE"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Freeze primary business model
# ------------------------------------------------------------

biz_primary_model = (
    biz_model_ranking.loc[0, "Model"]
)

biz_primary_performance = (
    biz_model_ranking.loc[0]
)

biz_primary_signal_column = (
    BUSINESS_SIGNAL_MAP[biz_primary_model]
)


# ------------------------------------------------------------
# 6. Create business risk signal
# ------------------------------------------------------------

biz_oos["Forecasted_Volatility"] = (
    biz_oos[biz_primary_signal_column]
)


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("=" * 80)
print("PRIMARY BUSINESS RISK SIGNAL")
print("=" * 80)

print(
    "Model-selection period:",
    biz_model_selection_data["Date"].min().date(),
    "to",
    biz_model_selection_data["Date"].max().date()
)

print(
    "Selected model :",
    biz_primary_model
)

print(
    "Signal column  :",
    biz_primary_signal_column
)

print(
    "Development MAE:",
    f"{biz_primary_performance['MAE']:.6f}"
)

print(
    "Development RMSE:",
    f"{biz_primary_performance['RMSE']:.6f}"
)

print(
    "Development observations:",
    int(biz_primary_performance["N"])
)

print()
print("Model ranking by development-period RMSE:")

display(
    biz_model_ranking[
        [
            "Model",
            "MAE",
            "RMSE",
            "N"
        ]
    ]
)

print()
print(
    "Primary model selection status: FROZEN"
)

print(
    "2026 observations were not used for model selection."
)

PRIMARY BUSINESS RISK SIGNAL
Model-selection period: 2024-01-01 to 2025-12-31
Selected model : EGARCH_t_VIX
Signal column  : EGARCH_t_VIX
Development MAE: 0.003730
Development RMSE: 0.005297
Development observations: 495

Model ranking by development-period RMSE:


,Model,MAE,RMSE,N
0,EGARCH_t_VIX,0.003730,0.005297,495
1,Historical_20D_VIX,0.003856,0.005633,495
2,EWMA_VIX,0.003859,0.005649,495
3,EGARCH_t,0.004585,0.005810,495
4,GARCH_t,0.004820,0.006334,495
5,GJR_GARCH_t,0.004846,0.006422,495
6,EWMA,0.004917,0.006476,495
7,Historical_20D,0.004913,0.006654,495



Primary model selection status: FROZEN
2026 observations were not used for model selection.


# Research Output Metadata & Information-Set Audit

In [4]:
# ============================================================
# INSPECT AVAILABLE PRE-OOS INFORMATION
# ============================================================

print("=" * 80)
print("PRE-OOS INFORMATION AVAILABLE FOR BUSINESS POLICY DESIGN")
print("=" * 80)

print("Research version:")
print(
    biz_research_metadata.get(
        "research_version"
    )
)

print()
print("Information-set rule:")
print(
    biz_research_metadata.get(
        "information_set"
    )
)

print()
print("OOS period:")
print(
    biz_research_metadata.get("oos_start"),
    "to",
    biz_research_metadata.get("oos_end")
)

print()
print("Authoritative model sources:")
for model, source in (
    biz_research_metadata
    .get("authoritative_sources", {})
    .items()
):
    print(f"{model:25s} -> {source}")

print()
print(
    "Important: Research Output v1 intentionally contains "
    "the frozen OOS forecast layer."
)

print(
    "The business policy must therefore be established "
    "without using the final OOS period to tune thresholds."
)

PRE-OOS INFORMATION AVAILABLE FOR BUSINESS POLICY DESIGN
Research version:
v1

Information-set rule:
Strict pre-return information set

OOS period:
2024-01-01 to 2026-09-22

Authoritative model sources:
Historical_20D            -> historical_20d_strict
EWMA                      -> test_data_vix_timing['EWMA_OneStep_Forecast']
GARCH_t                   -> test_data_vix_timing['GARCH_T_Volatility']
GJR_GARCH_t               -> test_data_vix_timing['GJR_GARCH_T_Volatility']
EGARCH_t                  -> test_data_vix_timing['EGARCH_T_Volatility']
Historical_20D_VIX        -> oos_hist_vix['Historical_20D_VIX_WalkForward']
EWMA_VIX                  -> ewma_vix_forecast
EGARCH_t_VIX              -> egarch_vix

Important: Research Output v1 intentionally contains the frozen OOS forecast layer.
The business policy must therefore be established without using the final OOS period to tune thresholds.


In [5]:
# ============================================================
# BUSINESS POLICY DEVELOPMENT / EVALUATION SPLIT
# ============================================================

# ------------------------------------------------------------
# 1. Create development and evaluation datasets
# ------------------------------------------------------------

biz_development = (
    biz_oos[
        biz_oos["Date"] <= BUSINESS_DEVELOPMENT_END
    ]
    .copy()
)

biz_evaluation = (
    biz_oos[
        biz_oos["Date"] >= BUSINESS_EVALUATION_START
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Validate the split
# ------------------------------------------------------------

if biz_development.empty:
    raise ValueError(
        "Business development period is empty."
    )

if biz_evaluation.empty:
    raise ValueError(
        "Business evaluation period is empty."
    )


if (
    biz_development["Date"].max()
    >=
    biz_evaluation["Date"].min()
):
    raise ValueError(
        "Business development and evaluation periods overlap."
    )


if (
    biz_development["Date"].max()
    != BUSINESS_DEVELOPMENT_END
):
    raise ValueError(
        "Unexpected business development end date."
    )


if (
    biz_evaluation["Date"].min()
    != BUSINESS_EVALUATION_START
):
    raise ValueError(
        "Unexpected business evaluation start date."
    )


# ------------------------------------------------------------
# 3. Report split
# ------------------------------------------------------------

print("=" * 80)
print("BUSINESS POLICY DEVELOPMENT / EVALUATION SPLIT")
print("=" * 80)

print()
print("Development period:")
print(
    biz_development["Date"].min().date(),
    "to",
    biz_development["Date"].max().date()
)
print("Observations:", len(biz_development))

print()
print("Evaluation period:")
print(
    biz_evaluation["Date"].min().date(),
    "to",
    biz_evaluation["Date"].max().date()
)
print("Observations:", len(biz_evaluation))

print()
print(
    "Policy development uses 2024–2025 only."
)

print(
    "Business impact evaluation uses 2026 only."
)

print()
print("Status: VALID")

BUSINESS POLICY DEVELOPMENT / EVALUATION SPLIT

Development period:
2024-01-01 to 2025-12-31
Observations: 495

Evaluation period:
2026-01-01 to 2026-09-22
Observations: 178

Policy development uses 2024–2025 only.
Business impact evaluation uses 2026 only.

Status: VALID


In [6]:
# ============================================================
# CALIBRATE BUSINESS RISK REGIME THRESHOLDS
# ============================================================

REGIME_PERCENTILES = {
    "Low": 0.25,
    "Normal": 0.50,
    "Elevated": 0.75,
    "High": 0.90,
}


# ------------------------------------------------------------
# 1. Development-period forecast distribution
# ------------------------------------------------------------

biz_development_volatility = (
    biz_development["Forecasted_Volatility"]
    .dropna()
)


if len(biz_development_volatility) == 0:
    raise ValueError(
        "No development-period volatility forecasts available."
    )


# ------------------------------------------------------------
# 2. Calculate thresholds
# ------------------------------------------------------------

biz_regime_thresholds = {
    regime: biz_development_volatility.quantile(percentile)
    for regime, percentile
    in REGIME_PERCENTILES.items()
}


# ------------------------------------------------------------
# 3. Validate threshold ordering
# ------------------------------------------------------------

threshold_values = list(
    biz_regime_thresholds.values()
)

if not all(
    threshold_values[i]
    < threshold_values[i + 1]
    for i in range(len(threshold_values) - 1)
):
    raise ValueError(
        "Risk-regime thresholds are not strictly increasing."
    )


# ------------------------------------------------------------
# 4. Display calibrated thresholds
# ------------------------------------------------------------

print("=" * 80)
print("BUSINESS RISK-REGIME THRESHOLDS")
print("=" * 80)

print()
print("Calibration period:")
print(
    biz_development["Date"].min().date(),
    "to",
    biz_development["Date"].max().date()
)

print()
print(
    "Forecast observations used:",
    len(biz_development_volatility)
)

print()
print("Frozen percentile thresholds:")

for regime, threshold in biz_regime_thresholds.items():

    percentile = (
        REGIME_PERCENTILES[regime] * 100
    )

    print(
        f"{regime:10s} "
        f"({percentile:>5.1f}th percentile): "
        f"{threshold:.6f}"
    )

print()
print(
    "Threshold policy status: CALIBRATED"
)

BUSINESS RISK-REGIME THRESHOLDS

Calibration period:
2024-01-01 to 2025-12-31

Forecast observations used: 495

Frozen percentile thresholds:
Low        ( 25.0th percentile): 0.004341
Normal     ( 50.0th percentile): 0.005549
Elevated   ( 75.0th percentile): 0.006882
High       ( 90.0th percentile): 0.007676

Threshold policy status: CALIBRATED


In [7]:
# ============================================================
# REVIEW DEVELOPMENT-PERIOD RISK REGIME DISTRIBUTION
# ============================================================

def assign_risk_regime(
    volatility,
    thresholds
):
    if volatility <= thresholds["Low"]:
        return "Low"

    elif volatility <= thresholds["Normal"]:
        return "Normal"

    elif volatility <= thresholds["Elevated"]:
        return "Elevated"

    elif volatility <= thresholds["High"]:
        return "High"

    else:
        return "Extreme"


# ------------------------------------------------------------
# 1. Assign regimes to development period
# ------------------------------------------------------------

biz_development["Risk_Regime"] = (
    biz_development["Forecasted_Volatility"]
    .apply(
        lambda x: assign_risk_regime(
            x,
            biz_regime_thresholds
        )
    )
)


# ------------------------------------------------------------
# 2. Calculate regime counts
# ------------------------------------------------------------

biz_regime_distribution = (
    biz_development["Risk_Regime"]
    .value_counts()
    .reindex(
        [
            "Low",
            "Normal",
            "Elevated",
            "High",
            "Extreme"
        ],
        fill_value=0
    )
    .to_frame("Observations")
)


# ------------------------------------------------------------
# 3. Calculate percentages
# ------------------------------------------------------------

biz_regime_distribution["Percentage"] = (
    biz_regime_distribution["Observations"]
    / len(biz_development)
    * 100
)


# ------------------------------------------------------------
# 4. Display
# ------------------------------------------------------------

print("=" * 80)
print("DEVELOPMENT-PERIOD RISK REGIME DISTRIBUTION")
print("=" * 80)

display(
    biz_regime_distribution
)

print(
    "Total observations:",
    biz_regime_distribution["Observations"].sum()
)

DEVELOPMENT-PERIOD RISK REGIME DISTRIBUTION


,Observations,Percentage
Risk_Regime,,
Low,124,25.050505
Normal,124,25.050505
Elevated,123,24.848485
High,74,14.949495
Extreme,50,10.101010


Total observations: 495


In [8]:
# ============================================================
# FREEZE BUSINESS EXPOSURE POLICY
# ============================================================

biz_exposure_policy = {
    "Low": 1.00,
    "Normal": 1.00,
    "Elevated": 0.80,
    "High": 0.60,
    "Extreme": 0.40,
}


# ------------------------------------------------------------
# 1. Validate policy completeness
# ------------------------------------------------------------

expected_regimes = [
    "Low",
    "Normal",
    "Elevated",
    "High",
    "Extreme"
]

if set(biz_exposure_policy.keys()) != set(expected_regimes):
    raise ValueError(
        "Exposure policy does not contain exactly "
        "the five required risk regimes."
    )


# ------------------------------------------------------------
# 2. Validate exposure bounds
# ------------------------------------------------------------

for regime, exposure in biz_exposure_policy.items():

    if not 0 < exposure <= 1:
        raise ValueError(
            f"Invalid exposure for {regime}: {exposure}"
        )


# ------------------------------------------------------------
# 3. Validate monotonic risk reduction
# ------------------------------------------------------------

ordered_exposures = [
    biz_exposure_policy[regime]
    for regime in expected_regimes
]

if not all(
    ordered_exposures[i]
    >= ordered_exposures[i + 1]
    for i in range(len(ordered_exposures) - 1)
):
    raise ValueError(
        "Exposure policy is not monotonically "
        "decreasing with risk."
    )


# ------------------------------------------------------------
# 4. Display frozen policy
# ------------------------------------------------------------

biz_exposure_policy_table = pd.DataFrame(
    {
        "Risk_Regime": expected_regimes,
        "Portfolio_Exposure": [
            biz_exposure_policy[regime]
            for regime in expected_regimes
        ]
    }
)

print("=" * 80)
print("FROZEN BUSINESS EXPOSURE POLICY")
print("=" * 80)

display(
    biz_exposure_policy_table
)

print()
print("Policy status: FROZEN")
print()
print(
    "The exposure policy will not be optimized "
    "using the 2026 evaluation period."
)

FROZEN BUSINESS EXPOSURE POLICY


,Risk_Regime,Portfolio_Exposure
0,Low,1.0
1,Normal,1.0
2,Elevated,0.8
3,High,0.6
4,Extreme,0.4



Policy status: FROZEN

The exposure policy will not be optimized using the 2026 evaluation period.


In [9]:
# ============================================================
# APPLY FROZEN POLICY TO 2026 EVALUATION PERIOD
# ============================================================

# Work on an independent copy
biz_evaluation = biz_evaluation.copy()


# ------------------------------------------------------------
# 1. Assign risk regime using FROZEN thresholds
# ------------------------------------------------------------

biz_evaluation["Risk_Regime"] = (
    biz_evaluation["Forecasted_Volatility"]
    .apply(
        lambda x: assign_risk_regime(
            x,
            biz_regime_thresholds
        )
    )
)


# ------------------------------------------------------------
# 2. Assign portfolio exposure using FROZEN policy
# ------------------------------------------------------------

biz_evaluation["Portfolio_Exposure"] = (
    biz_evaluation["Risk_Regime"]
    .map(biz_exposure_policy)
)


# ------------------------------------------------------------
# 3. Validate all regimes were mapped
# ------------------------------------------------------------

if biz_evaluation["Risk_Regime"].isna().any():
    raise ValueError(
        "Some 2026 observations have no risk regime."
    )


if biz_evaluation["Portfolio_Exposure"].isna().any():
    raise ValueError(
        "Some 2026 observations have no portfolio exposure."
    )


# ------------------------------------------------------------
# 4. Validate exposure values
# ------------------------------------------------------------

if not biz_evaluation["Portfolio_Exposure"].isin(
    biz_exposure_policy.values()
).all():
    raise ValueError(
        "Unexpected portfolio exposure detected."
    )


# ------------------------------------------------------------
# 5. Display regime distribution
# ------------------------------------------------------------

biz_2026_regime_distribution = (
    biz_evaluation["Risk_Regime"]
    .value_counts()
    .reindex(
        [
            "Low",
            "Normal",
            "Elevated",
            "High",
            "Extreme"
        ],
        fill_value=0
    )
    .to_frame("Observations")
)

biz_2026_regime_distribution["Percentage"] = (
    biz_2026_regime_distribution["Observations"]
    / len(biz_evaluation)
    * 100
)


# ------------------------------------------------------------
# 6. Display result
# ------------------------------------------------------------

print("=" * 80)
print("2026 BUSINESS RISK REGIME ASSIGNMENT")
print("=" * 80)

print()
print(
    "Evaluation period:",
    biz_evaluation["Date"].min().date(),
    "to",
    biz_evaluation["Date"].max().date()
)

print(
    "Observations:",
    len(biz_evaluation)
)

print()
print("2026 regime distribution:")

display(
    biz_2026_regime_distribution
)

print()
print("Exposure mapping:")

display(
    biz_evaluation[
        [
            "Date",
            "Forecasted_Volatility",
            "Risk_Regime",
            "Portfolio_Exposure"
        ]
    ].head(10)
)

print()
print("Policy application status: VALID")

2026 BUSINESS RISK REGIME ASSIGNMENT

Evaluation period: 2026-01-01 to 2026-09-22
Observations: 178

2026 regime distribution:


,Observations,Percentage
Risk_Regime,,
Low,24,13.483146
Normal,48,26.966292
Elevated,45,25.280899
High,11,6.179775
Extreme,50,28.089888



Exposure mapping:


,Date,Forecasted_Volatility,Risk_Regime,Portfolio_Exposure
495,2026-01-01,0.003037,Low,1.0
496,2026-01-02,0.002936,Low,1.0
497,2026-01-05,0.003062,Low,1.0
498,2026-01-06,0.003320,Low,1.0
499,2026-01-07,0.003301,Low,1.0
500,2026-01-08,0.004049,Low,1.0
501,2026-01-09,0.004681,Normal,1.0
502,2026-01-12,0.004529,Normal,1.0
503,2026-01-13,0.004621,Normal,1.0
504,2026-01-14,0.004571,Normal,1.0



Policy application status: VALID


In [10]:
# ============================================================
# CALCULATE STATIC VS ADAPTIVE SIMPLE PORTFOLIO RETURNS
# ============================================================

biz_evaluation = biz_evaluation.copy()


# ------------------------------------------------------------
# 1. Convert research log returns to simple returns
# ------------------------------------------------------------

biz_evaluation["NIFTY_Simple_Return"] = (
    np.exp(biz_evaluation["NIFTY_Log_Return"])
    - 1
)


# ------------------------------------------------------------
# 2. Static benchmark
# ------------------------------------------------------------

biz_evaluation["Static_Exposure"] = 1.00

biz_evaluation["Static_Portfolio_Return"] = (
    biz_evaluation["NIFTY_Simple_Return"]
    * biz_evaluation["Static_Exposure"]
)


# ------------------------------------------------------------
# 3. Adaptive portfolio
# ------------------------------------------------------------

biz_evaluation["Adaptive_Portfolio_Return"] = (
    biz_evaluation["NIFTY_Simple_Return"]
    * biz_evaluation["Portfolio_Exposure"]
)


# ------------------------------------------------------------
# 4. Validate calculations
# ------------------------------------------------------------

required_return_columns = [
    "NIFTY_Simple_Return",
    "Static_Portfolio_Return",
    "Adaptive_Portfolio_Return"
]

if (
    biz_evaluation[required_return_columns]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "Portfolio return calculation produced missing values."
    )


if not np.isfinite(
    biz_evaluation[required_return_columns].to_numpy()
).all():
    raise ValueError(
        "Portfolio return calculation produced non-finite values."
    )


# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("=" * 80)
print("STATIC VS ADAPTIVE PORTFOLIO SIMPLE RETURNS")
print("=" * 80)

display(
    biz_evaluation[
        [
            "Date",
            "NIFTY_Log_Return",
            "NIFTY_Simple_Return",
            "Risk_Regime",
            "Portfolio_Exposure",
            "Static_Portfolio_Return",
            "Adaptive_Portfolio_Return"
        ]
    ].head(10)
)


print()
print(
    "Evaluation observations:",
    len(biz_evaluation)
)

print(
    "Adaptive exposure levels:",
    sorted(
        biz_evaluation["Portfolio_Exposure"].unique()
    )
)

print()
print("Portfolio return calculation: VALID")

STATIC VS ADAPTIVE PORTFOLIO SIMPLE RETURNS


,Date,NIFTY_Log_Return,NIFTY_Simple_Return,Risk_Regime,Portfolio_Exposure,Static_Portfolio_Return,Adaptive_Portfolio_Return
495,2026-01-01,0.000649,0.000649,Low,1.0,0.000649,0.000649
496,2026-01-02,0.006937,0.006961,Low,1.0,0.006961,0.006961
497,2026-01-05,-0.002976,-0.002972,Low,1.0,-0.002972,-0.002972
498,2026-01-06,-0.002731,-0.002728,Low,1.0,-0.002728,-0.002728
499,2026-01-07,-0.001451,-0.001450,Low,1.0,-0.001450,-0.001450
500,2026-01-08,-0.010147,-0.010095,Low,1.0,-0.010095,-0.010095
501,2026-01-09,-0.007508,-0.007480,Normal,1.0,-0.007480,-0.007480
502,2026-01-12,0.004156,0.004164,Normal,1.0,0.004164,0.004164
503,2026-01-13,-0.002249,-0.002247,Normal,1.0,-0.002247,-0.002247
504,2026-01-14,-0.002595,-0.002592,Normal,1.0,-0.002592,-0.002592



Evaluation observations: 178
Adaptive exposure levels: [np.float64(0.4), np.float64(0.6), np.float64(0.8), np.float64(1.0)]

Portfolio return calculation: VALID


In [11]:
# ============================================================
# CALCULATE STATIC VS ADAPTIVE PORTFOLIO WEALTH
# ============================================================

biz_evaluation = biz_evaluation.copy()

# 1. Compound simple portfolio returns
biz_evaluation["Static_Wealth"] = (
    1 + biz_evaluation["Static_Portfolio_Return"]
).cumprod()

biz_evaluation["Adaptive_Wealth"] = (
    1 + biz_evaluation["Adaptive_Portfolio_Return"]
).cumprod()

# 2. Validate wealth series
wealth_columns = [
    "Static_Wealth",
    "Adaptive_Wealth"
]

if biz_evaluation[wealth_columns].isna().any().any():
    raise ValueError(
        "Portfolio wealth calculation produced missing values."
    )

if not np.isfinite(
    biz_evaluation[wealth_columns].to_numpy()
).all():
    raise ValueError(
        "Portfolio wealth calculation produced non-finite values."
    )

# Wealth should remain positive for these portfolios
if (biz_evaluation[wealth_columns] <= 0).any().any():
    raise ValueError(
        "Portfolio wealth contains non-positive values."
    )

# 3. Display
print("=" * 80)
print("STATIC VS ADAPTIVE PORTFOLIO WEALTH")
print("=" * 80)

display(
    biz_evaluation[
        [
            "Date",
            "Static_Portfolio_Return",
            "Adaptive_Portfolio_Return",
            "Static_Wealth",
            "Adaptive_Wealth"
        ]
    ].head(10)
)

print()
print("Evaluation observations:", len(biz_evaluation))

print(
    "Final static wealth:",
    round(biz_evaluation["Static_Wealth"].iloc[-1], 6)
)

print(
    "Final adaptive wealth:",
    round(biz_evaluation["Adaptive_Wealth"].iloc[-1], 6)
)

print()
print("Portfolio wealth calculation: VALID")

STATIC VS ADAPTIVE PORTFOLIO WEALTH


,Date,Static_Portfolio_Return,Adaptive_Portfolio_Return,Static_Wealth,Adaptive_Wealth
495,2026-01-01,0.000649,0.000649,1.000649,1.000649
496,2026-01-02,0.006961,0.006961,1.007614,1.007614
497,2026-01-05,-0.002972,-0.002972,1.004619,1.004619
498,2026-01-06,-0.002728,-0.002728,1.001879,1.001879
499,2026-01-07,-0.001450,-0.001450,1.000427,1.000427
500,2026-01-08,-0.010095,-0.010095,0.990327,0.990327
501,2026-01-09,-0.007480,-0.007480,0.982920,0.982920
502,2026-01-12,0.004164,0.004164,0.987013,0.987013
503,2026-01-13,-0.002247,-0.002247,0.984795,0.984795
504,2026-01-14,-0.002592,-0.002592,0.982242,0.982242



Evaluation observations: 178
Final static wealth: 0.892819
Final adaptive wealth: 0.939462

Portfolio wealth calculation: VALID


In [12]:
# ============================================================
# CALCULATE CORRECTED BUSINESS PERFORMANCE METRICS
# ============================================================

TRADING_DAYS = 252
RISK_FREE_RATE = 0.00

# ------------------------------------------------------------
# 1. Helper function for maximum drawdown
# ------------------------------------------------------------

def biz_max_drawdown(wealth_series):
    running_peak = wealth_series.cummax()
    drawdown = (wealth_series / running_peak) - 1
    return drawdown.min()


# ------------------------------------------------------------
# 2. Extract return series
# ------------------------------------------------------------

biz_static_returns = (
    biz_evaluation["Static_Portfolio_Return"]
)

biz_adaptive_returns = (
    biz_evaluation["Adaptive_Portfolio_Return"]
)


# ------------------------------------------------------------
# 3. Total returns
# ------------------------------------------------------------

biz_static_total_return = (
    biz_evaluation["Static_Wealth"].iloc[-1] - 1
)

biz_adaptive_total_return = (
    biz_evaluation["Adaptive_Wealth"].iloc[-1] - 1
)


# ------------------------------------------------------------
# 4. Annualized returns
# ------------------------------------------------------------

biz_n_obs = len(biz_evaluation)

biz_static_annualized_return = (
    biz_evaluation["Static_Wealth"].iloc[-1]
    ** (TRADING_DAYS / biz_n_obs)
    - 1
)

biz_adaptive_annualized_return = (
    biz_evaluation["Adaptive_Wealth"].iloc[-1]
    ** (TRADING_DAYS / biz_n_obs)
    - 1
)


# ------------------------------------------------------------
# 5. Annualized volatility
# ------------------------------------------------------------

biz_static_annualized_volatility = (
    biz_static_returns.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_annualized_volatility = (
    biz_adaptive_returns.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)


# ------------------------------------------------------------
# 6. Maximum drawdown
# ------------------------------------------------------------

biz_static_max_drawdown = biz_max_drawdown(
    biz_evaluation["Static_Wealth"]
)

biz_adaptive_max_drawdown = biz_max_drawdown(
    biz_evaluation["Adaptive_Wealth"]
)


# ------------------------------------------------------------
# 7. Sharpe ratio
# ------------------------------------------------------------

biz_static_daily_excess = (
    biz_static_returns
    - RISK_FREE_RATE / TRADING_DAYS
)

biz_adaptive_daily_excess = (
    biz_adaptive_returns
    - RISK_FREE_RATE / TRADING_DAYS
)

biz_static_sharpe = (
    biz_static_daily_excess.mean()
    / biz_static_daily_excess.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_sharpe = (
    biz_adaptive_daily_excess.mean()
    / biz_adaptive_daily_excess.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)


# ------------------------------------------------------------
# 8. Sortino ratio
# ------------------------------------------------------------

biz_static_downside = (
    biz_static_daily_excess[
        biz_static_daily_excess < 0
    ]
)

biz_adaptive_downside = (
    biz_adaptive_daily_excess[
        biz_adaptive_daily_excess < 0
    ]
)

biz_static_downside_deviation = (
    np.sqrt(
        np.mean(
            biz_static_downside ** 2
        )
    )
)

biz_adaptive_downside_deviation = (
    np.sqrt(
        np.mean(
            biz_adaptive_downside ** 2
        )
    )
)

biz_static_sortino = (
    biz_static_daily_excess.mean()
    / biz_static_downside_deviation
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_sortino = (
    biz_adaptive_daily_excess.mean()
    / biz_adaptive_downside_deviation
    * np.sqrt(TRADING_DAYS)
)


# ------------------------------------------------------------
# 9. Create business performance table
# ------------------------------------------------------------

biz_performance = pd.DataFrame(
    {
        "Metric": [
            "Observations",
            "Total Return",
            "Annualized Return",
            "Annualized Volatility",
            "Maximum Drawdown",
            "Sharpe Ratio",
            "Sortino Ratio"
        ],
        "Static": [
            biz_n_obs,
            biz_static_total_return,
            biz_static_annualized_return,
            biz_static_annualized_volatility,
            biz_static_max_drawdown,
            biz_static_sharpe,
            biz_static_sortino
        ],
        "Adaptive": [
            biz_n_obs,
            biz_adaptive_total_return,
            biz_adaptive_annualized_return,
            biz_adaptive_annualized_volatility,
            biz_adaptive_max_drawdown,
            biz_adaptive_sharpe,
            biz_adaptive_sortino
        ]
    }
)


# ------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------

print("=" * 80)
print("CORRECTED BUSINESS PERFORMANCE — 2026 EVALUATION")
print("=" * 80)

display(biz_performance)

print()
print("Evaluation period:",
      biz_evaluation["Date"].min(),
      "to",
      biz_evaluation["Date"].max())

print("Observations:", biz_n_obs)
print("Risk-free rate:", RISK_FREE_RATE)
print("Annualization basis:", TRADING_DAYS, "trading days")

print()
print("BUSINESS PERFORMANCE CALCULATION: VALID")

CORRECTED BUSINESS PERFORMANCE — 2026 EVALUATION


,Metric,Static,Adaptive
0,Observations,178.000000,178.000000
1,Total Return,-0.107181,-0.060538
2,Annualized Return,-0.148285,-0.084614
3,Annualized Volatility,0.147233,0.090845
4,Maximum Drawdown,-0.151818,-0.084730
5,Sharpe Ratio,-1.016548,-0.927889
6,Sortino Ratio,-0.976299,-0.921647



Evaluation period: 2026-01-01 00:00:00 to 2026-09-22 00:00:00
Observations: 178
Risk-free rate: 0.0
Annualization basis: 252 trading days

BUSINESS PERFORMANCE CALCULATION: VALID


In [13]:
# ============================================================
# QUANTIFY GROSS ADAPTIVE BUSINESS IMPACT
# ============================================================

biz_static_vol = biz_static_annualized_volatility
biz_adaptive_vol = biz_adaptive_annualized_volatility

biz_static_dd = biz_static_max_drawdown
biz_adaptive_dd = biz_adaptive_max_drawdown

biz_static_return = biz_static_total_return
biz_adaptive_return = biz_adaptive_total_return

# ------------------------------------------------------------
# Absolute differences
# ------------------------------------------------------------

biz_return_advantage = (
    biz_adaptive_return
    - biz_static_return
)

biz_volatility_reduction = (
    1
    - biz_adaptive_vol / biz_static_vol
)

biz_drawdown_reduction = (
    1
    - abs(biz_adaptive_dd) / abs(biz_static_dd)
)

biz_sharpe_difference = (
    biz_adaptive_sharpe
    - biz_static_sharpe
)

biz_sortino_difference = (
    biz_adaptive_sortino
    - biz_static_sortino
)

# ------------------------------------------------------------
# Gross business impact table
# ------------------------------------------------------------

biz_impact = pd.DataFrame(
    {
        "Impact_Metric": [
            "Compounded return advantage vs static",
            "Annualized return difference",
            "Annualized volatility reduction",
            "Maximum drawdown reduction",
            "Sharpe difference",
            "Sortino difference"
        ],
        "Value": [
            biz_return_advantage,
            (
                biz_adaptive_annualized_return
                - biz_static_annualized_return
            ),
            biz_volatility_reduction,
            biz_drawdown_reduction,
            biz_sharpe_difference,
            biz_sortino_difference
        ]
    }
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if biz_adaptive_vol >= biz_static_vol:
    print(
        "WARNING: Adaptive volatility is not lower "
        "than static volatility."
    )

if abs(biz_adaptive_dd) >= abs(biz_static_dd):
    print(
        "WARNING: Adaptive maximum drawdown is not lower "
        "than static maximum drawdown."
    )

if not np.isfinite(
    biz_impact["Value"].to_numpy()
).all():
    raise ValueError(
        "Business impact calculation produced non-finite values."
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("GROSS ADAPTIVE BUSINESS IMPACT — 2026 EVALUATION")
print("=" * 80)

display(biz_impact)

print()
print(
    "Compounded return advantage vs static:",
    f"{biz_return_advantage:.4%}"
)

print(
    "Annualized volatility reduction:",
    f"{biz_volatility_reduction:.2%}"
)

print(
    "Maximum drawdown reduction:",
    f"{biz_drawdown_reduction:.2%}"
)

print(
    "Sharpe difference:",
    f"{biz_sharpe_difference:.4f}"
)

print(
    "Sortino difference:",
    f"{biz_sortino_difference:.4f}"
)

print()
print("GROSS BUSINESS IMPACT CALCULATION: VALID")

GROSS ADAPTIVE BUSINESS IMPACT — 2026 EVALUATION


,Impact_Metric,Value
0,Compounded return advantage vs static,0.046643
1,Annualized return difference,0.063671
2,Annualized volatility reduction,0.382986
3,Maximum drawdown reduction,0.441900
4,Sharpe difference,0.088658
5,Sortino difference,0.054652



Compounded return advantage vs static: 4.6643%
Annualized volatility reduction: 38.30%
Maximum drawdown reduction: 44.19%
Sharpe difference: 0.0887
Sortino difference: 0.0547

GROSS BUSINESS IMPACT CALCULATION: VALID


In [14]:
# ============================================================
# MEASURE EXPOSURE CHANGES AND PORTFOLIO TURNOVER
# ============================================================

biz_evaluation = biz_evaluation.copy()

# ------------------------------------------------------------
# 1. Identify exposure changes
# ------------------------------------------------------------

biz_evaluation["Exposure_Change"] = (
    biz_evaluation["Portfolio_Exposure"]
    .diff()
    .fillna(0)
)

biz_evaluation["Absolute_Exposure_Change"] = (
    biz_evaluation["Exposure_Change"]
    .abs()
)

biz_evaluation["Exposure_Changed"] = (
    biz_evaluation["Exposure_Change"] != 0
)

# ------------------------------------------------------------
# 2. Turnover
# ------------------------------------------------------------

biz_total_turnover = (
    biz_evaluation["Absolute_Exposure_Change"].sum()
)

biz_average_daily_turnover = (
    biz_evaluation["Absolute_Exposure_Change"].mean()
)

biz_exposure_changes = (
    int(biz_evaluation["Exposure_Changed"].sum())
)

# ------------------------------------------------------------
# 3. Regime changes
# ------------------------------------------------------------

biz_evaluation["Regime_Changed"] = (
    biz_evaluation["Risk_Regime"]
    != biz_evaluation["Risk_Regime"].shift(1)
)

# First observation is not a genuine regime transition
biz_evaluation.loc[
    biz_evaluation.index[0],
    "Regime_Changed"
] = False

biz_regime_changes = (
    int(biz_evaluation["Regime_Changed"].sum())
)

# ------------------------------------------------------------
# 4. Exposure change distribution
# ------------------------------------------------------------

biz_exposure_change_distribution = (
    biz_evaluation.loc[
        biz_evaluation["Exposure_Changed"],
        [
            "Date",
            "Risk_Regime",
            "Portfolio_Exposure",
            "Exposure_Change"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

if biz_total_turnover < 0:
    raise ValueError(
        "Turnover cannot be negative."
    )

if biz_exposure_changes > len(biz_evaluation):
    raise ValueError(
        "Exposure-change count exceeds observations."
    )

if not np.isfinite(
    [
        biz_total_turnover,
        biz_average_daily_turnover,
        biz_exposure_changes,
        biz_regime_changes
    ]
).all():
    raise ValueError(
        "Turnover analysis produced non-finite values."
    )

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("=" * 80)
print("EXPOSURE CHANGES AND TURNOVER — 2026 EVALUATION")
print("=" * 80)

print(
    "Exposure changes:",
    biz_exposure_changes
)

print(
    "Regime changes:",
    biz_regime_changes
)

print(
    "Cumulative absolute exposure turnover:",
    round(biz_total_turnover, 4)
)

print(
    "Average daily absolute exposure change:",
    round(biz_average_daily_turnover, 6)
)

print()
print("Exposure-change events:")

display(
    biz_exposure_change_distribution
)

print()
print("TURNOVER ANALYSIS: VALID")

EXPOSURE CHANGES AND TURNOVER — 2026 EVALUATION
Exposure changes: 32
Regime changes: 44
Cumulative absolute exposure turnover: 6.4
Average daily absolute exposure change: 0.035955

Exposure-change events:


,Date,Risk_Regime,Portfolio_Exposure,Exposure_Change
507,2026-01-20,Elevated,0.8,-0.2
513,2026-01-29,Normal,1.0,0.2
514,2026-01-30,Elevated,0.8,-0.2
519,2026-02-06,Normal,1.0,0.2
524,2026-02-13,Elevated,0.8,-0.2
527,2026-02-18,Normal,1.0,0.2
528,2026-02-19,Elevated,0.8,-0.2
535,2026-03-02,High,0.6,-0.2
536,2026-03-04,Extreme,0.4,-0.2
577,2026-05-07,High,0.6,0.2



TURNOVER ANALYSIS: VALID


In [15]:
# ============================================================
# TRANSACTION-COST SENSITIVITY ANALYSIS
# ============================================================

# Scenario assumptions:
# Cost is applied to each unit of absolute exposure traded.
# These are illustrative sensitivity assumptions, not observed
# market transaction costs.

biz_transaction_cost_rates = {
    "5 bps": 0.0005,
    "10 bps": 0.0010,
    "20 bps": 0.0020
}

biz_turnover = (
    biz_evaluation["Absolute_Exposure_Change"].sum()
)

biz_static_compounded_return = (
    biz_evaluation["Static_Wealth"].iloc[-1] - 1
)

biz_cost_sensitivity = []

for biz_cost_label, biz_cost_rate in (
    biz_transaction_cost_rates.items()
):

    biz_daily_scenario_cost = (
        biz_evaluation["Absolute_Exposure_Change"]
        * biz_cost_rate
    )

    biz_daily_net_return = (
        biz_evaluation["Adaptive_Portfolio_Return"]
        - biz_daily_scenario_cost
    )

    biz_scenario_wealth = (
        1 + biz_daily_net_return
    ).cumprod()

    biz_adaptive_compounded_return_after_cost = (
        biz_scenario_wealth.iloc[-1] - 1
    )

    biz_advantage_after_cost = (
        biz_adaptive_compounded_return_after_cost
        - biz_static_compounded_return
    )

    biz_total_scenario_cost = (
        biz_daily_scenario_cost.sum()
    )

    biz_cost_sensitivity.append(
        {
            "Transaction_Cost": biz_cost_label,
            "Cost_Rate": biz_cost_rate,
            "Cumulative_Turnover": biz_turnover,
            "Estimated_Cumulative_Cost":
                biz_total_scenario_cost,
            "Adaptive_Compounded_Return_After_Cost":
                biz_adaptive_compounded_return_after_cost,
            "Advantage_vs_Static_After_Cost":
                biz_advantage_after_cost
        }
    )

biz_cost_sensitivity = pd.DataFrame(
    biz_cost_sensitivity
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if not np.isfinite(
    biz_cost_sensitivity[
        [
            "Estimated_Cumulative_Cost",
            "Adaptive_Compounded_Return_After_Cost",
            "Advantage_vs_Static_After_Cost"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Transaction-cost sensitivity produced "
        "non-finite values."
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("TRANSACTION-COST SENSITIVITY — 2026 EVALUATION")
print("=" * 80)

print(
    "Cumulative absolute exposure turnover:",
    round(biz_turnover, 4)
)

print()
print(
    "Scenario assumptions:",
    "5, 10 and 20 bps per unit of exposure traded."
)

display(
    biz_cost_sensitivity
)

print()
print("TRANSACTION-COST SENSITIVITY: VALID")

TRANSACTION-COST SENSITIVITY — 2026 EVALUATION
Cumulative absolute exposure turnover: 6.4

Scenario assumptions: 5, 10 and 20 bps per unit of exposure traded.


,Transaction_Cost,Cost_Rate,Cumulative_Turnover,Estimated_Cumulative_Cost,Adaptive_Compounded_Return_After_Cost,Advantage_vs_Static_After_Cost
0,5 bps,0.0005,6.4,0.0032,-0.063548,0.043633
1,10 bps,0.0010,6.4,0.0064,-0.066549,0.040632
2,20 bps,0.0020,6.4,0.0128,-0.072523,0.034658



TRANSACTION-COST SENSITIVITY: VALID


In [16]:
# ============================================================
# DAILY NET-OF-COST ADAPTIVE PORTFOLIO SIMULATION
# ============================================================

biz_evaluation = biz_evaluation.copy()

# ------------------------------------------------------------
# Central transaction-cost scenario
# ------------------------------------------------------------

biz_central_cost_rate = 0.0010  # 10 bps

# ------------------------------------------------------------
# 1. Daily transaction cost
# ------------------------------------------------------------

biz_evaluation["Daily_Transaction_Cost"] = (
    biz_evaluation["Absolute_Exposure_Change"]
    * biz_central_cost_rate
)

# ------------------------------------------------------------
# 2. Net adaptive portfolio return
# ------------------------------------------------------------

biz_evaluation["Adaptive_Net_Return"] = (
    biz_evaluation["Adaptive_Portfolio_Return"]
    - biz_evaluation["Daily_Transaction_Cost"]
)

# ------------------------------------------------------------
# 3. Net adaptive wealth
# ------------------------------------------------------------

biz_evaluation["Adaptive_Net_Wealth"] = (
    1 + biz_evaluation["Adaptive_Net_Return"]
).cumprod()

# ------------------------------------------------------------
# 4. Validate
# ------------------------------------------------------------

net_columns = [
    "Daily_Transaction_Cost",
    "Adaptive_Net_Return",
    "Adaptive_Net_Wealth"
]

if biz_evaluation[net_columns].isna().any().any():
    raise ValueError(
        "Net portfolio calculation produced missing values."
    )

if not np.isfinite(
    biz_evaluation[net_columns].to_numpy()
).all():
    raise ValueError(
        "Net portfolio calculation produced non-finite values."
    )

if (
    biz_evaluation["Adaptive_Net_Wealth"] <= 0
).any():
    raise ValueError(
        "Net adaptive wealth contains non-positive values."
    )

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

biz_total_transaction_cost = (
    biz_evaluation["Daily_Transaction_Cost"].sum()
)

biz_final_net_wealth = (
    biz_evaluation["Adaptive_Net_Wealth"].iloc[-1]
)

biz_final_net_return = (
    biz_final_net_wealth - 1
)

print("=" * 80)
print("DAILY NET-OF-COST ADAPTIVE PORTFOLIO — 10 BPS SCENARIO")
print("=" * 80)

print(
    "Transaction-cost rate:",
    f"{biz_central_cost_rate:.2%}"
)

print(
    "Total transaction cost:",
    f"{biz_total_transaction_cost:.4%}"
)

print(
    "Final net wealth:",
    round(biz_final_net_wealth, 6)
)

print(
    "Final net return:",
    f"{biz_final_net_return:.4%}"
)

print()
print("DAILY NET-OF-COST SIMULATION: VALID")

DAILY NET-OF-COST ADAPTIVE PORTFOLIO — 10 BPS SCENARIO
Transaction-cost rate: 0.10%
Total transaction cost: 0.6400%
Final net wealth: 0.933451
Final net return: -6.6549%

DAILY NET-OF-COST SIMULATION: VALID


In [17]:
# ============================================================
# NET-OF-COST BUSINESS PERFORMANCE
# ============================================================

# ------------------------------------------------------------
# 1. Net adaptive return series
# ------------------------------------------------------------

biz_net_adaptive_returns = (
    biz_evaluation["Adaptive_Net_Return"]
)

biz_static_returns = (
    biz_evaluation["Static_Portfolio_Return"]
)

# ------------------------------------------------------------
# 2. Total returns
# ------------------------------------------------------------

biz_static_total_return = (
    biz_evaluation["Static_Wealth"].iloc[-1] - 1
)

biz_adaptive_net_total_return = (
    biz_evaluation["Adaptive_Net_Wealth"].iloc[-1] - 1
)

# ------------------------------------------------------------
# 3. Annualized returns
# ------------------------------------------------------------

biz_n_obs = len(biz_evaluation)

biz_static_annualized_return = (
    biz_evaluation["Static_Wealth"].iloc[-1]
    ** (TRADING_DAYS / biz_n_obs)
    - 1
)

biz_adaptive_net_annualized_return = (
    biz_evaluation["Adaptive_Net_Wealth"].iloc[-1]
    ** (TRADING_DAYS / biz_n_obs)
    - 1
)

# ------------------------------------------------------------
# 4. Annualized volatility
# ------------------------------------------------------------

biz_static_annualized_volatility = (
    biz_static_returns.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_net_annualized_volatility = (
    biz_net_adaptive_returns.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# 5. Maximum drawdown
# ------------------------------------------------------------

biz_static_max_drawdown_net = biz_max_drawdown(
    biz_evaluation["Static_Wealth"]
)

biz_adaptive_net_max_drawdown = biz_max_drawdown(
    biz_evaluation["Adaptive_Net_Wealth"]
)

# ------------------------------------------------------------
# 6. Sharpe ratio
# ------------------------------------------------------------

biz_static_excess_net = (
    biz_static_returns
    - RISK_FREE_RATE / TRADING_DAYS
)

biz_adaptive_net_excess = (
    biz_net_adaptive_returns
    - RISK_FREE_RATE / TRADING_DAYS
)

biz_static_sharpe_net = (
    biz_static_excess_net.mean()
    / biz_static_excess_net.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_net_sharpe = (
    biz_adaptive_net_excess.mean()
    / biz_adaptive_net_excess.std(ddof=1)
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# 7. Sortino ratio
# ------------------------------------------------------------

biz_static_downside_net = (
    biz_static_excess_net[
        biz_static_excess_net < 0
    ]
)

biz_adaptive_net_downside = (
    biz_adaptive_net_excess[
        biz_adaptive_net_excess < 0
    ]
)

biz_static_downside_deviation_net = np.sqrt(
    np.mean(
        biz_static_downside_net ** 2
    )
)

biz_adaptive_net_downside_deviation = np.sqrt(
    np.mean(
        biz_adaptive_net_downside ** 2
    )
)

biz_static_sortino_net = (
    biz_static_excess_net.mean()
    / biz_static_downside_deviation_net
    * np.sqrt(TRADING_DAYS)
)

biz_adaptive_net_sortino = (
    biz_adaptive_net_excess.mean()
    / biz_adaptive_net_downside_deviation
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# 8. Performance table
# ------------------------------------------------------------

biz_net_performance = pd.DataFrame(
    {
        "Metric": [
            "Observations",
            "Total Return",
            "Annualized Return",
            "Annualized Volatility",
            "Maximum Drawdown",
            "Sharpe Ratio",
            "Sortino Ratio"
        ],
        "Static": [
            biz_n_obs,
            biz_static_total_return,
            biz_static_annualized_return,
            biz_static_annualized_volatility,
            biz_static_max_drawdown_net,
            biz_static_sharpe_net,
            biz_static_sortino_net
        ],
        "Adaptive_Net_10bps": [
            biz_n_obs,
            biz_adaptive_net_total_return,
            biz_adaptive_net_annualized_return,
            biz_adaptive_net_annualized_volatility,
            biz_adaptive_net_max_drawdown,
            biz_adaptive_net_sharpe,
            biz_adaptive_net_sortino
        ]
    }
)

# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

if not np.isfinite(
    biz_net_performance[
        ["Static", "Adaptive_Net_10bps"]
    ].to_numpy()
).all():
    raise ValueError(
        "Net-of-cost performance contains non-finite values."
    )

print("=" * 80)
print("NET-OF-COST BUSINESS PERFORMANCE — 2026 EVALUATION")
print("=" * 80)

display(biz_net_performance)

print()
print(
    "Transaction-cost scenario:",
    f"{biz_central_cost_rate:.2%}"
)

print(
    "Evaluation period:",
    biz_evaluation["Date"].min(),
    "to",
    biz_evaluation["Date"].max()
)

print()
print("NET-OF-COST PERFORMANCE CALCULATION: VALID")

NET-OF-COST BUSINESS PERFORMANCE — 2026 EVALUATION


,Metric,Static,Adaptive_Net_10bps
0,Observations,178.000000,178.000000
1,Total Return,-0.107181,-0.066549
2,Annualized Return,-0.148285,-0.092895
3,Annualized Volatility,0.147233,0.091103
4,Maximum Drawdown,-0.151818,-0.086383
5,Sharpe Ratio,-1.016548,-1.024714
6,Sortino Ratio,-0.976299,-1.017631



Transaction-cost scenario: 0.10%
Evaluation period: 2026-01-01 00:00:00 to 2026-09-22 00:00:00

NET-OF-COST PERFORMANCE CALCULATION: VALID


In [18]:
# ============================================================
# FINAL NET BUSINESS IMPACT — 10 BPS SCENARIO
# ============================================================

# ------------------------------------------------------------
# 1. Net return advantage
# ------------------------------------------------------------

biz_net_return_advantage = (
    biz_adaptive_net_total_return
    - biz_static_total_return
)

biz_net_annualized_return_advantage = (
    biz_adaptive_net_annualized_return
    - biz_static_annualized_return
)

# ------------------------------------------------------------
# 2. Risk reductions
# ------------------------------------------------------------

biz_net_volatility_reduction = (
    1
    - (
        biz_adaptive_net_annualized_volatility
        / biz_static_annualized_volatility
    )
)

biz_net_drawdown_reduction = (
    1
    - (
        abs(biz_adaptive_net_max_drawdown)
        / abs(biz_static_max_drawdown)
    )
)

# ------------------------------------------------------------
# 3. Risk-adjusted performance differences
# ------------------------------------------------------------

biz_net_sharpe_difference = (
    biz_adaptive_net_sharpe
    - biz_static_sharpe_net
)

biz_net_sortino_difference = (
    biz_adaptive_net_sortino
    - biz_static_sortino_net
)

# ------------------------------------------------------------
# 4. Final business-impact table
# ------------------------------------------------------------

biz_final_net_impact = pd.DataFrame(
    {
        "Business_Impact": [
            "Total return advantage",
            "Annualized return advantage",
            "Annualized volatility reduction",
            "Maximum drawdown reduction",
            "Sharpe difference",
            "Sortino difference"
        ],
        "Value": [
            biz_net_return_advantage,
            biz_net_annualized_return_advantage,
            biz_net_volatility_reduction,
            biz_net_drawdown_reduction,
            biz_net_sharpe_difference,
            biz_net_sortino_difference
        ]
    }
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

if not np.isfinite(
    biz_final_net_impact["Value"].to_numpy()
).all():
    raise ValueError(
        "Final net business impact contains non-finite values."
    )

print("=" * 80)
print("FINAL NET BUSINESS IMPACT — 10 BPS TRANSACTION-COST SCENARIO")
print("=" * 80)

display(biz_final_net_impact)

print()
print(
    "Net total-return advantage:",
    f"{biz_net_return_advantage:.4%}"
)

print(
    "Net annualized-return advantage:",
    f"{biz_net_annualized_return_advantage:.4%}"
)

print(
    "Net annualized-volatility reduction:",
    f"{biz_net_volatility_reduction:.2%}"
)

print(
    "Net maximum-drawdown reduction:",
    f"{biz_net_drawdown_reduction:.2%}"
)

print(
    "Sharpe difference:",
    f"{biz_net_sharpe_difference:.4f}"
)

print(
    "Sortino difference:",
    f"{biz_net_sortino_difference:.4f}"
)

print()
print("FINAL NET BUSINESS IMPACT: VALID")

FINAL NET BUSINESS IMPACT — 10 BPS TRANSACTION-COST SCENARIO


,Business_Impact,Value
0,Total return advantage,0.040632
1,Annualized return advantage,0.055390
2,Annualized volatility reduction,0.381232
3,Maximum drawdown reduction,0.431008
4,Sharpe difference,-0.008166
5,Sortino difference,-0.041332



Net total-return advantage: 4.0632%
Net annualized-return advantage: 5.5390%
Net annualized-volatility reduction: 38.12%
Net maximum-drawdown reduction: 43.10%
Sharpe difference: -0.0082
Sortino difference: -0.0413

FINAL NET BUSINESS IMPACT: VALID


In [19]:
# ============================================================
# REGIME-LEVEL BUSINESS IMPACT ANALYSIS
# ============================================================

biz_regime_analysis = (
    biz_evaluation
    .groupby("Risk_Regime", observed=True)
    .agg(
        Observations=("Date", "count"),
        Average_Forecasted_Volatility=(
            "Forecasted_Volatility",
            "mean"
        ),
        Average_Exposure=(
            "Portfolio_Exposure",
            "mean"
        ),
        Sum_Daily_Static_Return=(
            "Static_Portfolio_Return",
            "sum"
        ),
        Sum_Daily_Adaptive_Net_Return=(
            "Adaptive_Net_Return",
            "sum"
        ),
        Average_Daily_Static_Return=(
            "Static_Portfolio_Return",
            "mean"
        ),
        Average_Daily_Adaptive_Net_Return=(
            "Adaptive_Net_Return",
            "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# Regime ordering
# ------------------------------------------------------------

biz_regime_order = [
    "Low",
    "Normal",
    "Elevated",
    "High",
    "Extreme"
]

biz_regime_analysis["Risk_Regime"] = pd.Categorical(
    biz_regime_analysis["Risk_Regime"],
    categories=biz_regime_order,
    ordered=True
)

biz_regime_analysis = (
    biz_regime_analysis
    .sort_values("Risk_Regime")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Return advantage by regime
# ------------------------------------------------------------

biz_regime_analysis["Difference_Summed_Daily_Returns"] = (
    biz_regime_analysis["Sum_Daily_Adaptive_Net_Return"]
    - biz_regime_analysis["Sum_Daily_Static_Return"]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if biz_regime_analysis["Observations"].sum() != len(
    biz_evaluation
):
    raise ValueError(
        "Regime observations do not reconcile with "
        "the evaluation sample."
    )

if not np.isfinite(
    biz_regime_analysis[
        [
            "Average_Forecasted_Volatility",
            "Average_Exposure",
            "Sum_Daily_Static_Return",
            "Sum_Daily_Adaptive_Net_Return",
            "Difference_Summed_Daily_Returns"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Regime analysis contains non-finite values."
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 80)
print("REGIME-LEVEL BUSINESS IMPACT — 2026 EVALUATION")
print("=" * 80)

display(
    biz_regime_analysis[
        [
            "Risk_Regime",
            "Observations",
            "Average_Forecasted_Volatility",
            "Average_Exposure",
            "Sum_Daily_Static_Return",
            "Sum_Daily_Adaptive_Net_Return",
            "Difference_Summed_Daily_Returns"
        ]
    ]
)

print()
print(
    "Total observations:",
    biz_regime_analysis["Observations"].sum()
)

print()
print("REGIME-LEVEL ANALYSIS: VALID")

REGIME-LEVEL BUSINESS IMPACT — 2026 EVALUATION


,Risk_Regime,Observations,Average_Forecasted_Volatility,Average_Exposure,Sum_Daily_Static_Return,Sum_Daily_Adaptive_Net_Return,Difference_Summed_Daily_Returns
0,Low,24,0.003951,1.0,-0.021429,-0.021429,0.000000
1,Normal,48,0.004909,1.0,0.019894,0.017894,-0.002000
2,Elevated,45,0.006089,0.8,-0.023286,-0.021229,0.002057
3,High,11,0.007260,0.6,-0.035089,-0.022254,0.012836
4,Extreme,50,0.010983,0.4,-0.045808,-0.018923,0.026885



Total observations: 178

REGIME-LEVEL ANALYSIS: VALID


In [20]:
# ============================================================
# HIGH/EXTREME REGIME PROTECTION ANALYSIS
# ============================================================

biz_protection_regimes = [
    "High",
    "Extreme"
]

biz_protection_data = biz_evaluation[
    biz_evaluation["Risk_Regime"].isin(
        biz_protection_regimes
    )
].copy()

# ------------------------------------------------------------
# 1. Observations and exposure
# ------------------------------------------------------------

biz_protection_observations = len(
    biz_protection_data
)

biz_protection_share = (
    biz_protection_observations
    / len(biz_evaluation)
)

biz_average_protection_exposure = (
    biz_protection_data[
        "Portfolio_Exposure"
    ].mean()
)

# ------------------------------------------------------------
# 2. Static vs adaptive net return
# ------------------------------------------------------------

biz_protection_static_return = (
    biz_protection_data[
        "Static_Portfolio_Return"
    ].sum()
)

biz_protection_adaptive_net_return = (
    biz_protection_data[
        "Adaptive_Net_Return"
    ].sum()
)

biz_protection_return_advantage = (
    biz_protection_adaptive_net_return
    - biz_protection_static_return
)

# ------------------------------------------------------------
# 3. Loss-magnitude improvement
# ------------------------------------------------------------

biz_protection_loss_magnitude_improvement = (
    abs(biz_protection_static_return)
    - abs(biz_protection_adaptive_net_return)
)

# ------------------------------------------------------------
# 4. Exposure reduction
# ------------------------------------------------------------

biz_protection_exposure_reduction = (
    1
    - biz_average_protection_exposure
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

if (
    biz_protection_observations
    <= 0
):
    raise ValueError(
        "No High/Extreme observations found."
    )

if not np.isfinite(
    [
        biz_protection_share,
        biz_average_protection_exposure,
        biz_protection_static_return,
        biz_protection_adaptive_net_return,
        biz_protection_return_advantage,
        biz_protection_loss_magnitude_improvement,
        biz_protection_exposure_reduction
    ]
).all():
    raise ValueError(
        "Protection analysis produced non-finite values."
    )

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("=" * 80)
print("HIGH/EXTREME REGIME PROTECTION — 2026 EVALUATION")
print("=" * 80)

print(
    "High + Extreme observations:",
    biz_protection_observations
)

print(
    "Share of evaluation period:",
    f"{biz_protection_share:.2%}"
)

print(
    "Average exposure:",
    f"{biz_average_protection_exposure:.2%}"
)

print(
    "Exposure reduction:",
    f"{biz_protection_exposure_reduction:.2%}"
)


print(
    "Static sum of daily returns:",
    f"{biz_protection_static_return:.4%}"
)

print(
    "Difference in summed daily returns:",
    f"{biz_protection_return_advantage:.4%}"
)

print(
    "Loss-magnitude improvement:",
    f"{biz_protection_loss_magnitude_improvement:.4%}"
)

print()
print("HIGH/EXTREME PROTECTION ANALYSIS: VALID")

HIGH/EXTREME REGIME PROTECTION — 2026 EVALUATION
High + Extreme observations: 61
Share of evaluation period: 34.27%
Average exposure: 43.61%
Exposure reduction: 56.39%
Static sum of daily returns: -8.0897%
Difference in summed daily returns: 3.9721%
Loss-magnitude improvement: 3.9721%

HIGH/EXTREME PROTECTION ANALYSIS: VALID


### 21. Business Evaluation State and Dependency Check

Before proceeding to the subsequent decision-impact analyses, the business evaluation dataset is subjected to a structural and dependency validation check.

This checkpoint confirms that `biz_evaluation` is available as a valid DataFrame, contains the required business-evaluation variables, has no missing values in the fields used for downstream analysis, and satisfies basic exposure and return-integrity conditions.

The business evaluation uses the held-out 2026 period. Primary business-performance conclusions are based on compounded portfolio performance, while arithmetic sums of daily returns are retained only as secondary diagnostic measures.

The validation is also intended to improve notebook reproducibility by reducing dependence on accidental persistence of variables in the Python kernel.

The required fields include:

* `Risk_Regime`
* `Portfolio_Exposure`
* `Static_Portfolio_Return`
* `Adaptive_Net_Return`

The validation does not modify the evaluation dataset or recalculate model outputs. Its purpose is to provide a reproducible checkpoint before subsequent analytical sections.

In [21]:
# ============================================================
# BUSINESS EVALUATION STATE & DEPENDENCY CHECK
# ============================================================
print("=" * 80)
print("BUSINESS EVALUATION STATE & DEPENDENCY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Confirm primary evaluation object
# ------------------------------------------------------------

if "biz_evaluation" not in globals():
    raise NameError(
        "biz_evaluation is not available. "
        "Re-establish the business evaluation dataset before continuing."
    )

if not isinstance(biz_evaluation, pd.DataFrame):
    raise TypeError(
        "biz_evaluation exists but is not a pandas DataFrame."
    )

# ------------------------------------------------------------
# 2. Confirm required columns
# ------------------------------------------------------------

required_biz_columns = [
    "Risk_Regime",
    "Portfolio_Exposure",
    "Static_Portfolio_Return",
    "Adaptive_Net_Return"
]

missing_biz_columns = [
    col
    for col in required_biz_columns
    if col not in biz_evaluation.columns
]

if missing_biz_columns:
    raise KeyError(
        "biz_evaluation is missing required columns: "
        f"{missing_biz_columns}"
    )

# ------------------------------------------------------------
# 3. Basic structural checks
# ------------------------------------------------------------

if len(biz_evaluation) == 0:
    raise ValueError(
        "biz_evaluation exists but contains no observations."
    )

if biz_evaluation.index.has_duplicates:
    raise ValueError(
        "biz_evaluation contains duplicate index values."
    )

# ------------------------------------------------------------
# 4. Required-value checks
# ------------------------------------------------------------

required_value_data = biz_evaluation[
    required_biz_columns
].copy()

missing_required_values = (
    required_value_data.isna().sum()
)

if missing_required_values.any():
    print("\nMissing values in required columns:")
    print(
        missing_required_values[
            missing_required_values > 0
        ]
    )
else:
    print("\nRequired columns: no missing values")

# ------------------------------------------------------------
# 5. Regime coverage
# ------------------------------------------------------------

print("\nRisk regime distribution:")
print(
    biz_evaluation["Risk_Regime"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 6. Exposure sanity check
# ------------------------------------------------------------

exposure_min = biz_evaluation[
    "Portfolio_Exposure"
].min()

exposure_max = biz_evaluation[
    "Portfolio_Exposure"
].max()

print("\nPortfolio exposure range:")
print(
    f"Minimum: {exposure_min:.4%}"
)

print(
    f"Maximum: {exposure_max:.4%}"
)

if (
    exposure_min < 0
    or exposure_max > 1
):
    raise ValueError(
        "Portfolio_Exposure contains values outside [0, 1]."
    )

# ------------------------------------------------------------
# 7. Return sanity check
# ------------------------------------------------------------

if not np.isfinite(
    biz_evaluation[
        [
            "Static_Portfolio_Return",
            "Adaptive_Net_Return"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Static or adaptive returns contain non-finite values."
    )

# ------------------------------------------------------------
# 8. Final checkpoint
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BUSINESS EVALUATION STATE CHECK: PASS")
print("=" * 80)

print(
    "biz_evaluation shape:",
    biz_evaluation.shape
)

print(
    "Required columns verified:",
    len(required_biz_columns)
)

print(
    "High/Extreme observations:",
    biz_evaluation[
        "Risk_Regime"
    ].isin(["High", "Extreme"]).sum()
)

print(
    "Business evaluation state is valid and ready for downstream analysis."
)

BUSINESS EVALUATION STATE & DEPENDENCY CHECK

Required columns: no missing values

Risk regime distribution:
Risk_Regime
Elevated    45
Extreme     50
High        11
Low         24
Normal      48
Name: count, dtype: int64

Portfolio exposure range:
Minimum: 40.0000%
Maximum: 100.0000%

BUSINESS EVALUATION STATE CHECK: PASS
biz_evaluation shape: (178, 28)
Required columns verified: 4
High/Extreme observations: 61
Business evaluation state is valid and ready for downstream analysis.


### Business Impact Assessment

The preceding analysis established the relationship between forecasted volatility, risk regimes, and adaptive portfolio exposure during the held-out 2026 evaluation period.

The next stage translates these model outputs into a business decision perspective. The objective is to determine whether the adaptive exposure mechanism provides measurable benefits relative to a static-exposure benchmark while accounting for an illustrative transaction-cost assumption.

The business evaluation focuses on four practical questions:

1. **Does adaptive exposure management reduce the magnitude of portfolio losses over the evaluation period?**
2. **Does it reduce portfolio volatility and maximum drawdown?**
3. **Do the observed benefits remain after incorporating an illustrative transaction-cost assumption?**
4. **Are the benefits concentrated in higher-risk market regimes where exposure reduction is most relevant?**

Primary business-performance conclusions are based on compounded portfolio returns and wealth. Arithmetic sums of daily returns are retained as secondary diagnostics and are not interpreted as compounded portfolio returns.

The analysis is intended to demonstrate how a volatility-forecasting and risk-regime framework can support practical decisions involving dynamic capital allocation, exposure management, risk-budget control, and downside-risk management.

The results should not be interpreted as evidence of guaranteed profitability or as a claim that the adaptive strategy universally outperforms a static strategy. Instead, they provide an empirical evaluation of whether the proposed risk-management mechanism produced measurable changes in portfolio risk and performance during the held-out evaluation period.

In [22]:
# ============================================================
# FULL EVALUATION BUSINESS IMPACT
# ============================================================

# ------------------------------------------------------------
# 1. Aggregate portfolio performance
# ------------------------------------------------------------

biz_total_static_return = (
    biz_evaluation[
        "Static_Portfolio_Return"
    ].sum()
)

biz_total_adaptive_net_return = (
    biz_evaluation[
        "Adaptive_Net_Return"
    ].sum()
)

# ------------------------------------------------------------
# 2. Business impact / return advantage
# ------------------------------------------------------------

biz_total_return_advantage = (
    biz_total_adaptive_net_return
    - biz_total_static_return
)

# ------------------------------------------------------------
# 3. Exposure statistics
# ------------------------------------------------------------

biz_average_exposure = (
    biz_evaluation[
        "Portfolio_Exposure"
    ].mean()
)

biz_exposure_reduction = (
    1
    - biz_average_exposure
)

# ------------------------------------------------------------
# 4. Downside observations
# ------------------------------------------------------------

biz_static_negative_days = (
    biz_evaluation[
        "Static_Portfolio_Return"
    ] < 0
).sum()

biz_adaptive_negative_days = (
    biz_evaluation[
        "Adaptive_Net_Return"
    ] < 0
).sum()

# ------------------------------------------------------------
# 5. Loss comparison
# ------------------------------------------------------------

biz_static_total_losses = (
    biz_evaluation.loc[
        biz_evaluation["Static_Portfolio_Return"] < 0,
        "Static_Portfolio_Return"
    ].sum()
)

biz_adaptive_total_losses = (
    biz_evaluation.loc[
        biz_evaluation["Adaptive_Net_Return"] < 0,
        "Adaptive_Net_Return"
    ].sum()
)

biz_loss_avoidance = (
    biz_adaptive_total_losses
    - biz_static_total_losses
)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

biz_business_metrics = np.array([
    biz_total_static_return,
    biz_total_adaptive_net_return,
    biz_total_return_advantage,
    biz_average_exposure,
    biz_exposure_reduction,
    biz_static_total_losses,
    biz_adaptive_total_losses,
    biz_loss_avoidance
])

if not np.isfinite(
    biz_business_metrics
).all():
    raise ValueError(
        "Business impact analysis produced non-finite values."
    )

if not (
    0 <= biz_average_exposure <= 1
):
    raise ValueError(
        "Average portfolio exposure is outside [0, 1]."
    )

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("=" * 80)
print("FULL EVALUATION — BUSINESS IMPACT ASSESSMENT")
print("=" * 80)

print(
    "Evaluation observations:",
    len(biz_evaluation)
)

print(
    "Static cumulative return:",
    f"{biz_total_static_return:.4%}"
)

print(
    "Adaptive cumulative net return:",
    f"{biz_total_adaptive_net_return:.4%}"
)

print(
    "Adaptive return advantage:",
    f"{biz_total_return_advantage:.4%}"
)

print(
    "Average portfolio exposure:",
    f"{biz_average_exposure:.2%}"
)

print(
    "Overall exposure reduction:",
    f"{biz_exposure_reduction:.2%}"
)

print(
    "Static negative-return observations:",
    biz_static_negative_days
)

print(
    "Adaptive negative-return observations:",
    biz_adaptive_negative_days
)

print(
    "Static cumulative losses:",
    f"{biz_static_total_losses:.4%}"
)

print(
    "Adaptive cumulative losses:",
    f"{biz_adaptive_total_losses:.4%}"
)

print(
    "Loss avoidance:",
    f"{biz_loss_avoidance:.4%}"
)

print()
print("FULL EVALUATION BUSINESS IMPACT: VALID")

FULL EVALUATION — BUSINESS IMPACT ASSESSMENT
Evaluation observations: 178
Static cumulative return: -10.5719%
Adaptive cumulative net return: -6.5941%
Adaptive return advantage: 3.9778%
Average portfolio exposure: 75.62%
Overall exposure reduction: 24.38%
Static negative-return observations: 91
Adaptive negative-return observations: 92
Static cumulative losses: -66.3554%
Adaptive cumulative losses: -43.4018%
Loss avoidance: 22.9536%

FULL EVALUATION BUSINESS IMPACT: VALID


# Return Definition and Compounding Validation

Before finalizing the business-impact metrics, the return construction used in the evaluation dataset is validated. The previous section reported arithmetic sums of daily portfolio returns; however, if the portfolio return fields represent simple daily returns, compounded performance provides the more appropriate measure of cumulative portfolio growth or loss.

This checkpoint therefore evaluates the return series for admissible values and calculates both arithmetic and compounded performance. The comparison allows subsequent business-impact analysis to distinguish between daily return contribution and actual cumulative portfolio performance.

The underlying evaluation dataset is not modified by this validation.

In [23]:
# ============================================================
# 23 — RETURN DEFINITION & COMPOUNDING VALIDATION
# ============================================================

print("=" * 80)
print("RETURN DEFINITION & COMPOUNDING VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Confirm required return columns
# ------------------------------------------------------------

required_return_columns = [
    "Static_Portfolio_Return",
    "Adaptive_Net_Return"
]

missing_return_columns = [
    col
    for col in required_return_columns
    if col not in biz_evaluation.columns
]

if missing_return_columns:
    raise KeyError(
        "Required return columns are missing: "
        f"{missing_return_columns}"
    )

# ------------------------------------------------------------
# 2. Extract return series
# ------------------------------------------------------------

biz_static_returns = (
    biz_evaluation[
        "Static_Portfolio_Return"
    ].astype(float)
)

biz_adaptive_returns = (
    biz_evaluation[
        "Adaptive_Net_Return"
    ].astype(float)
)

# ------------------------------------------------------------
# 3. Missing-value validation
# ------------------------------------------------------------

if (
    biz_static_returns.isna().any()
    or biz_adaptive_returns.isna().any()
):
    raise ValueError(
        "Return series contain missing values."
    )

# ------------------------------------------------------------
# 4. Finite-value validation
# ------------------------------------------------------------

if not np.isfinite(
    biz_static_returns.to_numpy()
).all():
    raise ValueError(
        "Static portfolio returns contain non-finite values."
    )

if not np.isfinite(
    biz_adaptive_returns.to_numpy()
).all():
    raise ValueError(
        "Adaptive portfolio returns contain non-finite values."
    )

# ------------------------------------------------------------
# 5. Basic simple-return admissibility check
# ------------------------------------------------------------

if (
    (biz_static_returns <= -1).any()
    or (biz_adaptive_returns <= -1).any()
):
    raise ValueError(
        "Return series contain a value <= -100%, "
        "which is incompatible with standard simple-return compounding."
    )

# ------------------------------------------------------------
# 6. Arithmetic return contribution
# ------------------------------------------------------------

biz_static_arithmetic_return = (
    biz_static_returns.sum()
)

biz_adaptive_arithmetic_return = (
    biz_adaptive_returns.sum()
)

biz_arithmetic_return_advantage = (
    biz_adaptive_arithmetic_return
    - biz_static_arithmetic_return
)

# ------------------------------------------------------------
# 7. Compounded cumulative return
# ------------------------------------------------------------

biz_static_compounded_return = (
    (1 + biz_static_returns).prod()
    - 1
)

biz_adaptive_compounded_return = (
    (1 + biz_adaptive_returns).prod()
    - 1
)

biz_compounded_return_advantage = (
    biz_adaptive_compounded_return
    - biz_static_compounded_return
)

# ------------------------------------------------------------
# 8. Wealth index
# ------------------------------------------------------------

biz_static_terminal_wealth = (
    (1 + biz_static_returns).prod()
)

biz_adaptive_terminal_wealth = (
    (1 + biz_adaptive_returns).prod()
)

# ------------------------------------------------------------
# 9. Validation of calculated metrics
# ------------------------------------------------------------

biz_return_validation_metrics = np.array([
    biz_static_arithmetic_return,
    biz_adaptive_arithmetic_return,
    biz_arithmetic_return_advantage,
    biz_static_compounded_return,
    biz_adaptive_compounded_return,
    biz_compounded_return_advantage,
    biz_static_terminal_wealth,
    biz_adaptive_terminal_wealth
])

if not np.isfinite(
    biz_return_validation_metrics
).all():
    raise ValueError(
        "Return validation produced non-finite results."
    )

# ------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------

print("\nReturn observations:")
print(
    "Static:",
    len(biz_static_returns)
)

print(
    "Adaptive:",
    len(biz_adaptive_returns)
)

print("\nArithmetic return contribution:")
print(
    "Static:",
    f"{biz_static_arithmetic_return:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_arithmetic_return:.4%}"
)

print(
    "Arithmetic advantage:",
    f"{biz_arithmetic_return_advantage:.4%}"
)

print("\nCompounded cumulative return:")
print(
    "Static:",
    f"{biz_static_compounded_return:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_compounded_return:.4%}"
)

print(
    "Compounded advantage:",
    f"{biz_compounded_return_advantage:.4%}"
)

print("\nTerminal wealth from 1.00 starting capital:")
print(
    "Static:",
    f"{biz_static_terminal_wealth:.6f}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_terminal_wealth:.6f}"
)

print("\n" + "=" * 80)
print("CELL 23 STATUS: PASS")
print("=" * 80)

print(
    "Return series are valid for simple-return compounding."
)

print(
    "Both arithmetic and compounded performance "
    "have been calculated without modifying biz_evaluation."
)

RETURN DEFINITION & COMPOUNDING VALIDATION

Return observations:
Static: 178
Adaptive: 178

Arithmetic return contribution:
Static: -10.5719%
Adaptive: -6.5941%
Arithmetic advantage: 3.9778%

Compounded cumulative return:
Static: -10.7181%
Adaptive: -6.6549%
Compounded advantage: 4.0632%

Terminal wealth from 1.00 starting capital:
Static: 0.892819
Adaptive: 0.933451

CELL 23 STATUS: PASS
Return series are valid for simple-return compounding.
Both arithmetic and compounded performance have been calculated without modifying biz_evaluation.


## Drawdown and Downside Protection Analysis

Cumulative return provides an overall measure of portfolio performance, but it does not fully describe the severity of losses experienced during the evaluation period. For a risk-management system, the magnitude of peak-to-trough capital decline is an important business-impact measure.

This section therefore evaluates maximum drawdown for the static and adaptive strategies using compounded portfolio wealth. The comparison assesses whether adaptive exposure management reduced the severity of capital declines during the out-of-sample evaluation period.

Maximum drawdown is used as a downside-risk indicator rather than as evidence of profitability. A lower drawdown would indicate that the adaptive framework provided stronger capital protection relative to the static benchmark.

In [24]:
# ============================================================
# MAXIMUM DRAWDOWN & DOWNSIDE PROTECTION
# ============================================================

print("=" * 80)
print("MAXIMUM DRAWDOWN & DOWNSIDE PROTECTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Construct compounded wealth paths
# ------------------------------------------------------------

biz_static_wealth = (
    (1 + biz_static_returns)
    .cumprod()
)

biz_adaptive_wealth = (
    (1 + biz_adaptive_returns)
    .cumprod()
)

# ------------------------------------------------------------
# 2. Construct running peaks
# ------------------------------------------------------------

biz_static_peak = (
    biz_static_wealth
    .cummax()
)

biz_adaptive_peak = (
    biz_adaptive_wealth
    .cummax()
)

# ------------------------------------------------------------
# 3. Calculate drawdown series
# ------------------------------------------------------------

biz_static_drawdown = (
    biz_static_wealth
    / biz_static_peak
    - 1
)

biz_adaptive_drawdown = (
    biz_adaptive_wealth
    / biz_adaptive_peak
    - 1
)

# ------------------------------------------------------------
# 4. Maximum drawdown
# ------------------------------------------------------------

biz_static_max_drawdown = (
    biz_static_drawdown.min()
)

biz_adaptive_max_drawdown = (
    biz_adaptive_drawdown.min()
)

# ------------------------------------------------------------
# 5. Drawdown reduction
# ------------------------------------------------------------

biz_drawdown_reduction = (
    abs(biz_static_max_drawdown)
    - abs(biz_adaptive_max_drawdown)
)

# ------------------------------------------------------------
# 6. Percentage improvement in drawdown severity
# ------------------------------------------------------------

if abs(biz_static_max_drawdown) > 0:

    biz_drawdown_improvement = (
        1
        -
        (
            abs(biz_adaptive_max_drawdown)
            /
            abs(biz_static_max_drawdown)
        )
    )

else:

    biz_drawdown_improvement = np.nan

# ------------------------------------------------------------
# 7. Worst wealth level
# ------------------------------------------------------------

biz_static_min_wealth = (
    biz_static_wealth.min()
)

biz_adaptive_min_wealth = (
    biz_adaptive_wealth.min()
)

# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

biz_drawdown_metrics = np.array([
    biz_static_max_drawdown,
    biz_adaptive_max_drawdown,
    biz_drawdown_reduction,
    biz_drawdown_improvement,
    biz_static_min_wealth,
    biz_adaptive_min_wealth
])

if not np.isfinite(
    biz_drawdown_metrics
).all():
    raise ValueError(
        "Drawdown analysis produced non-finite values."
    )

if (
    biz_static_min_wealth <= 0
    or biz_adaptive_min_wealth <= 0
):
    raise ValueError(
        "Wealth path contains a non-positive value."
    )

# ------------------------------------------------------------
# 9. Display
# ------------------------------------------------------------

print("\nMaximum drawdown:")
print(
    "Static strategy:",
    f"{biz_static_max_drawdown:.4%}"
)

print(
    "Adaptive strategy:",
    f"{biz_adaptive_max_drawdown:.4%}"
)

print(
    "Drawdown reduction:",
    f"{biz_drawdown_reduction:.4%}"
)

print(
    "Drawdown severity improvement:",
    f"{biz_drawdown_improvement:.2%}"
)

print("\nMinimum portfolio wealth:")
print(
    "Static:",
    f"{biz_static_min_wealth:.6f}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_min_wealth:.6f}"
)

print("\n" + "=" * 80)
print("CELL 24 STATUS: PASS")
print("=" * 80)

print(
    "Maximum drawdown calculated from compounded wealth paths."
)

print(
    "No underlying evaluation data were modified."
)

MAXIMUM DRAWDOWN & DOWNSIDE PROTECTION

Maximum drawdown:
Static strategy: -15.1818%
Adaptive strategy: -8.6383%
Drawdown reduction: 6.5435%
Drawdown severity improvement: 43.10%

Minimum portfolio wealth:
Static: 0.854640
Adaptive: 0.920573

CELL 24 STATUS: PASS
Maximum drawdown calculated from compounded wealth paths.
No underlying evaluation data were modified.


### Risk-Adjusted Performance Assessment

The preceding sections established that adaptive exposure management reduced maximum drawdown and produced a lower compounded portfolio loss than the static benchmark during the held-out evaluation period.

The next step evaluates whether these changes remained favorable after accounting for portfolio-return variability.

This section compares annualized return, annualized volatility, and Sharpe ratio for the static and adaptive strategies over the out-of-sample evaluation period.

The purpose is not to demonstrate absolute profitability. Instead, the analysis evaluates whether the adaptive framework changed the observed risk-return characteristics relative to maintaining static exposure.

Because the adaptive strategy incorporates exposure reduction, lower volatility does not necessarily imply a higher Sharpe ratio. The Sharpe ratio is therefore reported as an independent risk-adjusted metric rather than being assumed to improve.

In [25]:
# ============================================================
# RISK-ADJUSTED PERFORMANCE ASSESSMENT
# ============================================================

print("=" * 80)
print("RISK-ADJUSTED PERFORMANCE ASSESSMENT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Annualization convention
# ------------------------------------------------------------

biz_trading_days_per_year = 252

# ------------------------------------------------------------
# 2. Annualized return
# ------------------------------------------------------------

biz_static_annualized_return = (
    biz_static_terminal_wealth
    ** (
        biz_trading_days_per_year
        / len(biz_static_returns)
    )
    - 1
)

biz_adaptive_annualized_return = (
    biz_adaptive_terminal_wealth
    ** (
        biz_trading_days_per_year
        / len(biz_adaptive_returns)
    )
    - 1
)

# ------------------------------------------------------------
# 3. Annualized volatility
# ------------------------------------------------------------

biz_static_annualized_volatility = (
    biz_static_returns.std(ddof=1)
    * np.sqrt(biz_trading_days_per_year)
)

biz_adaptive_annualized_volatility = (
    biz_adaptive_returns.std(ddof=1)
    * np.sqrt(biz_trading_days_per_year)
)

# ------------------------------------------------------------
# 4. Sharpe ratio
# ------------------------------------------------------------
# Risk-free rate is set to zero for a direct strategy
# comparison. This avoids introducing an external assumption.

biz_static_sharpe = (
    biz_static_annualized_return
    / biz_static_annualized_volatility
)

biz_adaptive_sharpe = (
    biz_adaptive_annualized_return
    / biz_adaptive_annualized_volatility
)

# ------------------------------------------------------------
# 5. Risk-adjusted improvement
# ------------------------------------------------------------

biz_sharpe_improvement = (
    biz_adaptive_sharpe
    - biz_static_sharpe
)

biz_volatility_reduction = (
    1
    -
    (
        biz_adaptive_annualized_volatility
        /
        biz_static_annualized_volatility
    )
)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

biz_risk_adjusted_metrics = np.array([
    biz_static_annualized_return,
    biz_adaptive_annualized_return,
    biz_static_annualized_volatility,
    biz_adaptive_annualized_volatility,
    biz_static_sharpe,
    biz_adaptive_sharpe,
    biz_sharpe_improvement,
    biz_volatility_reduction
])

if not np.isfinite(
    biz_risk_adjusted_metrics
).all():
    raise ValueError(
        "Risk-adjusted performance produced non-finite values."
    )

if (
    biz_static_annualized_volatility <= 0
    or biz_adaptive_annualized_volatility <= 0
):
    raise ValueError(
        "Annualized volatility must be positive."
    )

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("\nAnnualized return:")
print(
    "Static:",
    f"{biz_static_annualized_return:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_annualized_return:.4%}"
)

print("\nAnnualized volatility:")
print(
    "Static:",
    f"{biz_static_annualized_volatility:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_annualized_volatility:.4%}"
)

print(
    "Volatility reduction:",
    f"{biz_volatility_reduction:.2%}"
)

print("\nSharpe ratio (risk-free rate = 0):")
print(
    "Static:",
    f"{biz_static_sharpe:.4f}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_sharpe:.4f}"
)

print(
    "Sharpe improvement:",
    f"{biz_sharpe_improvement:.4f}"
)

print("\n" + "=" * 80)
print("CELL 25 STATUS: PASS")
print("=" * 80)

print(
    "Risk-adjusted performance calculated successfully."
)

print(
    "Risk-free rate assumption: 0%."
)

print(
    "No underlying evaluation data were modified."
)

RISK-ADJUSTED PERFORMANCE ASSESSMENT

Annualized return:
Static: -14.8285%
Adaptive: -9.2895%

Annualized volatility:
Static: 14.7233%
Adaptive: 9.1103%
Volatility reduction: 38.12%

Sharpe ratio (risk-free rate = 0):
Static: -1.0071
Adaptive: -1.0197
Sharpe improvement: -0.0125

CELL 25 STATUS: PASS
Risk-adjusted performance calculated successfully.
Risk-free rate assumption: 0%.
No underlying evaluation data were modified.


#### Downside Volatility Assessment

Standard deviation treats positive and negative return fluctuations symmetrically. For a risk-management application, downside variability is more directly relevant because adverse return movements represent the primary source of portfolio risk.

This section therefore compares downside deviation between the static and adaptive strategies. The analysis considers only negative daily portfolio returns and evaluates whether adaptive exposure management reduced the variability associated with adverse market movements.

The result complements the maximum-drawdown analysis by measuring downside variability at the daily-return level.

In [26]:
# ============================================================
# DOWNSIDE VOLATILITY / SEMI-DEVIATION
# ============================================================

print("=" * 80)
print("DOWNSIDE VOLATILITY ASSESSMENT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Identify negative daily returns
# ------------------------------------------------------------

biz_static_downside_returns = (
    biz_static_returns[
        biz_static_returns < 0
    ]
)

biz_adaptive_downside_returns = (
    biz_adaptive_returns[
        biz_adaptive_returns < 0
    ]
)

# ------------------------------------------------------------
# 2. Calculate downside deviation
# ------------------------------------------------------------
# Measures the RMS magnitude of negative daily returns.
# Zero is used as the downside target.

biz_static_downside_deviation = np.sqrt(
    np.mean(
        biz_static_downside_returns ** 2
    )
)

biz_adaptive_downside_deviation = np.sqrt(
    np.mean(
        biz_adaptive_downside_returns ** 2
    )
)

# ------------------------------------------------------------
# 3. Annualize downside deviation
# ------------------------------------------------------------

biz_static_annualized_downside_deviation = (
    biz_static_downside_deviation
    * np.sqrt(biz_trading_days_per_year)
)

biz_adaptive_annualized_downside_deviation = (
    biz_adaptive_downside_deviation
    * np.sqrt(biz_trading_days_per_year)
)

# ------------------------------------------------------------
# 4. Downside-risk reduction
# ------------------------------------------------------------

biz_downside_volatility_reduction = (
    1
    -
    (
        biz_adaptive_annualized_downside_deviation
        /
        biz_static_annualized_downside_deviation
    )
)

# ------------------------------------------------------------
# 5. Average negative return
# ------------------------------------------------------------

biz_static_average_negative_return = (
    biz_static_downside_returns.mean()
)

biz_adaptive_average_negative_return = (
    biz_adaptive_downside_returns.mean()
)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

biz_downside_metrics = np.array([
    biz_static_downside_deviation,
    biz_adaptive_downside_deviation,
    biz_static_annualized_downside_deviation,
    biz_adaptive_annualized_downside_deviation,
    biz_downside_volatility_reduction,
    biz_static_average_negative_return,
    biz_adaptive_average_negative_return
])

if not np.isfinite(
    biz_downside_metrics
).all():
    raise ValueError(
        "Downside volatility analysis produced non-finite values."
    )

if (
    biz_static_downside_deviation <= 0
    or biz_adaptive_downside_deviation <= 0
):
    raise ValueError(
        "Downside deviation must be positive."
    )

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("\nNegative-return observations:")
print(
    "Static:",
    len(biz_static_downside_returns)
)

print(
    "Adaptive:",
    len(biz_adaptive_downside_returns)
)

print("\nAnnualized downside deviation:")
print(
    "Static:",
    f"{biz_static_annualized_downside_deviation:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_annualized_downside_deviation:.4%}"
)

print(
    "Downside volatility reduction:",
    f"{biz_downside_volatility_reduction:.2%}"
)

print("\nAverage negative daily return:")
print(
    "Static:",
    f"{biz_static_average_negative_return:.4%}"
)

print(
    "Adaptive:",
    f"{biz_adaptive_average_negative_return:.4%}"
)

print("\n" + "=" * 80)
print("CELL 26 STATUS: PASS")
print("=" * 80)

print(
    "Downside volatility calculated successfully."
)

print(
    "No underlying evaluation data were modified."
)

DOWNSIDE VOLATILITY ASSESSMENT

Negative-return observations:
Static: 91
Adaptive: 92

Annualized downside deviation:
Static: 15.3303%
Adaptive: 9.1737%
Downside volatility reduction: 40.16%

Average negative daily return:
Static: -0.7292%
Adaptive: -0.4718%

CELL 26 STATUS: PASS
Downside volatility calculated successfully.
No underlying evaluation data were modified.


#### Business Impact Scorecard

The preceding analyses evaluated the adaptive framework from multiple business-risk perspectives, including compounded portfolio performance, exposure reduction, maximum drawdown, return volatility, and downside variability.

This section consolidates the validated out-of-sample results into a single business-impact scorecard. The purpose is to distinguish between performance outcomes and risk-control benefits and to provide a concise set of decision-oriented KPIs for reporting and visualization.

The scorecard does not create a new performance measure or combine the individual metrics into an artificial overall score. Each KPI retains its original definition and interpretation.

Where transaction costs are relevant, the final business-performance conclusions use the illustrative 10 bps net-of-cost scenario established earlier in the analysis.

In [27]:
# ============================================================
# BUSINESS IMPACT SCORECARD
# ============================================================

print("=" * 80)
print("BUSINESS IMPACT SCORECARD")
print("=" * 80)

# ------------------------------------------------------------
# 1. Build consolidated scorecard
# ------------------------------------------------------------

biz_impact_scorecard = pd.DataFrame({
    "Business_Dimension": [
        "Compounded performance",
        "Return advantage",
        "Average exposure",
        "Overall exposure reduction",
        "Maximum drawdown",
        "Drawdown severity improvement",
        "Annualized volatility",
        "Volatility reduction",
        "Downside deviation",
        "Downside volatility reduction"
    ],

    "Static_Strategy": [
        biz_static_compounded_return,
        np.nan,
        1.0,
        np.nan,
        biz_static_max_drawdown,
        np.nan,
        biz_static_annualized_volatility,
        np.nan,
        biz_static_annualized_downside_deviation,
        np.nan
    ],

    "Adaptive_Strategy": [
        biz_adaptive_compounded_return,
        biz_compounded_return_advantage,
        biz_average_exposure,
        biz_exposure_reduction,
        biz_adaptive_max_drawdown,
        biz_drawdown_improvement,
        biz_adaptive_annualized_volatility,
        biz_volatility_reduction,
        biz_adaptive_annualized_downside_deviation,
        biz_downside_volatility_reduction
    ],

    "Business_Interpretation": [
        "Adaptive strategy reduced cumulative portfolio loss.",
        "Adaptive strategy outperformed static exposure.",
        "Adaptive system maintained lower average market exposure.",
        "Adaptive system reduced average exposure versus full exposure.",
        "Adaptive strategy experienced a smaller peak-to-trough decline.",
        "Maximum drawdown severity was materially lower.",
        "Adaptive strategy exhibited lower total return variability.",
        "Adaptive strategy reduced portfolio volatility.",
        "Adaptive strategy exhibited lower downside variability.",
        "Adaptive strategy reduced downside volatility."
    ]
})

# ------------------------------------------------------------
# 2. Validate scorecard
# ------------------------------------------------------------

if biz_impact_scorecard.empty:
    raise ValueError(
        "Business impact scorecard is empty."
    )

if (
    biz_impact_scorecard["Business_Dimension"]
    .duplicated()
    .any()
):
    raise ValueError(
        "Business impact scorecard contains duplicate dimensions."
    )

# ------------------------------------------------------------
# 3. Required source metrics validation
# ------------------------------------------------------------

scorecard_source_metrics = np.array([
    biz_static_compounded_return,
    biz_adaptive_compounded_return,
    biz_compounded_return_advantage,
    biz_average_exposure,
    biz_exposure_reduction,
    biz_static_max_drawdown,
    biz_adaptive_max_drawdown,
    biz_drawdown_improvement,
    biz_static_annualized_volatility,
    biz_adaptive_annualized_volatility,
    biz_volatility_reduction,
    biz_static_annualized_downside_deviation,
    biz_adaptive_annualized_downside_deviation,
    biz_downside_volatility_reduction
])

if not np.isfinite(
    scorecard_source_metrics
).all():
    raise ValueError(
        "One or more scorecard source metrics are non-finite."
    )

# ------------------------------------------------------------
# 4. Display scorecard
# ------------------------------------------------------------

print("\nBUSINESS IMPACT SCORECARD")
print("-" * 80)

print(
    biz_impact_scorecard.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 5. Key business outcomes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("KEY BUSINESS OUTCOMES")
print("=" * 80)

print(
    "Compounded performance improvement:",
    f"{biz_compounded_return_advantage:.4%}"
)

print(
    "Maximum drawdown severity improvement:",
    f"{biz_drawdown_improvement:.2%}"
)

print(
    "Overall exposure reduction:",
    f"{biz_exposure_reduction:.2%}"
)

print(
    "Annualized volatility reduction:",
    f"{biz_volatility_reduction:.2%}"
)

print(
    "Downside volatility reduction:",
    f"{biz_downside_volatility_reduction:.2%}"
)

print("\n" + "=" * 80)
print("CELL 27 STATUS: PASS")
print("=" * 80)

print(
    "Validated business-impact metrics consolidated successfully."
)

print(
    "No underlying evaluation data were modified."
)

BUSINESS IMPACT SCORECARD

BUSINESS IMPACT SCORECARD
--------------------------------------------------------------------------------
           Business_Dimension  Static_Strategy  Adaptive_Strategy                                         Business_Interpretation
       Compounded performance        -0.107181          -0.066549            Adaptive strategy reduced cumulative portfolio loss.
             Return advantage              NaN           0.040632                 Adaptive strategy outperformed static exposure.
             Average exposure         1.000000           0.756180       Adaptive system maintained lower average market exposure.
   Overall exposure reduction              NaN           0.243820  Adaptive system reduced average exposure versus full exposure.
             Maximum drawdown        -0.151818          -0.086383 Adaptive strategy experienced a smaller peak-to-trough decline.
Drawdown severity improvement              NaN           0.431008                 Maxi

#### Regime-Based Risk Decision Framework

The adaptive system translates estimated market-risk conditions into dynamic portfolio exposure. This section converts the observed risk regimes and exposure levels into an explicit business decision framework.

The objective is to demonstrate how the analytical system can support practical risk-management actions rather than functioning only as a forecasting model.

Each risk regime is associated with its observed average portfolio exposure and a corresponding decision posture. The resulting framework can support applications such as dynamic capital allocation, portfolio risk-budget management, and escalation of risk controls during adverse market conditions.

The decision labels are descriptive interpretations of the existing exposure policy and do not introduce a new trading strategy.

In [28]:
# ============================================================
# REGIME-BASED RISK DECISION FRAMEWORK
# ============================================================

print("=" * 80)
print("REGIME-BASED RISK DECISION FRAMEWORK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Regime order
# ------------------------------------------------------------

biz_regime_order = [
    "Low",
    "Normal",
    "Elevated",
    "High",
    "Extreme"
]

# ------------------------------------------------------------
# 2. Regime-level exposure summary
# ------------------------------------------------------------

biz_regime_decision = (
    biz_evaluation
    .groupby("Risk_Regime")
    .agg(
        Observations=(
            "Portfolio_Exposure",
            "size"
        ),
        Average_Exposure=(
            "Portfolio_Exposure",
            "mean"
        ),
        Minimum_Exposure=(
            "Portfolio_Exposure",
            "min"
        ),
        Maximum_Exposure=(
            "Portfolio_Exposure",
            "max"
        )
    )
    .reindex(biz_regime_order)
)

# ------------------------------------------------------------
# 3. Decision posture
# ------------------------------------------------------------

biz_decision_posture = {
    "Low": "Normal / full-risk posture",
    "Normal": "Standard allocation",
    "Elevated": "Cautious allocation",
    "High": "Defensive allocation",
    "Extreme": "Maximum risk reduction"
}

biz_regime_decision[
    "Decision_Posture"
] = (
    biz_regime_decision.index
    .map(biz_decision_posture)
)

# ------------------------------------------------------------
# 4. Exposure reduction relative to full exposure
# ------------------------------------------------------------

biz_regime_decision[
    "Exposure_Reduction"
] = (
    1
    - biz_regime_decision[
        "Average_Exposure"
    ]
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

if biz_regime_decision.empty:
    raise ValueError(
        "Regime decision table is empty."
    )

if (
    biz_regime_decision["Observations"]
    .isna()
    .any()
):
    raise ValueError(
        "One or more expected risk regimes are absent."
    )

if not np.isfinite(
    biz_regime_decision[
        [
            "Average_Exposure",
            "Minimum_Exposure",
            "Maximum_Exposure",
            "Exposure_Reduction"
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "Regime decision metrics contain non-finite values."
    )

if (
    (biz_regime_decision["Average_Exposure"] < 0)
    .any()
    or
    (biz_regime_decision["Average_Exposure"] > 1)
    .any()
):
    raise ValueError(
        "Regime exposure values fall outside [0, 1]."
    )

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("\nREGIME-LEVEL BUSINESS DECISION FRAMEWORK")
print("-" * 80)

display_table = biz_regime_decision.copy()

display_table["Average_Exposure"] = (
    display_table["Average_Exposure"]
    .map(lambda x: f"{x:.2%}")
)

display_table["Minimum_Exposure"] = (
    display_table["Minimum_Exposure"]
    .map(lambda x: f"{x:.2%}")
)

display_table["Maximum_Exposure"] = (
    display_table["Maximum_Exposure"]
    .map(lambda x: f"{x:.2%}")
)

display_table["Exposure_Reduction"] = (
    display_table["Exposure_Reduction"]
    .map(lambda x: f"{x:.2%}")
)

print(
    display_table.to_string()
)

# ------------------------------------------------------------
# 7. Final checkpoint
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 28 STATUS: PASS")
print("=" * 80)

print(
    "All five risk regimes mapped to observed exposure levels."
)

print(
    "Decision postures generated from the existing adaptive policy."
)

print(
    "No underlying evaluation data were modified."
)

REGIME-BASED RISK DECISION FRAMEWORK

REGIME-LEVEL BUSINESS DECISION FRAMEWORK
--------------------------------------------------------------------------------
             Observations Average_Exposure Minimum_Exposure Maximum_Exposure            Decision_Posture Exposure_Reduction
Risk_Regime                                                                                                                
Low                    24          100.00%          100.00%          100.00%  Normal / full-risk posture              0.00%
Normal                 48          100.00%          100.00%          100.00%         Standard allocation              0.00%
Elevated               45           80.00%           80.00%           80.00%         Cautious allocation             20.00%
High                   11           60.00%           60.00%           60.00%        Defensive allocation             40.00%
Extreme                50           40.00%           40.00%           40.00%      Maximum risk r

In [29]:
# ============================================================
# DIAGNOSTIC — BUSINESS EVALUATION DATA LINEAGE
# ============================================================

print("=" * 80)
print("BUSINESS EVALUATION DATA LINEAGE DIAGNOSTIC")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check whether key objects exist
# ------------------------------------------------------------

objects_to_check = [
    "biz_evaluation",
    "evaluation",
    "test_data_vix_timing",
    "test_data_vix_valid",
    "clean_nifty",
    "oos_forecast_matrix",
    "forecast_matrix"
]

print("\nOBJECT EXISTENCE")
print("-" * 80)

for obj_name in objects_to_check:
    exists = obj_name in globals()
    
    if exists:
        obj = globals()[obj_name]
        
        try:
            shape = obj.shape
        except AttributeError:
            shape = "N/A"
        
        print(
            f"{obj_name:<25} EXISTS | "
            f"type={type(obj).__name__:<15} | "
            f"shape={shape}"
        )
    else:
        print(
            f"{obj_name:<25} NOT FOUND"
        )

# ------------------------------------------------------------
# 2. Inspect biz_evaluation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BIZ_EVALUATION STRUCTURE")
print("=" * 80)

if "biz_evaluation" not in globals():
    raise NameError(
        "biz_evaluation does not exist in the current kernel."
    )

print(
    f"Shape: {biz_evaluation.shape}"
)

print("\nColumns:")
for i, column in enumerate(
    biz_evaluation.columns,
    start=1
):
    print(
        f"{i:>3}. {column}"
    )

# ------------------------------------------------------------
# 3. Date information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BIZ_EVALUATION DATE INFORMATION")
print("=" * 80)

if "Date" in biz_evaluation.columns:
    
    print(
        f"Minimum Date: {biz_evaluation['Date'].min()}"
    )
    
    print(
        f"Maximum Date: {biz_evaluation['Date'].max()}"
    )
    
    print(
        f"Unique Dates: {biz_evaluation['Date'].nunique()}"
    )

else:
    print(
        "No 'Date' column found."
    )

# ------------------------------------------------------------
# 4. Index information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INDEX INFORMATION")
print("=" * 80)

print(
    f"Index type: {type(biz_evaluation.index).__name__}"
)

print(
    f"Index unique: {biz_evaluation.index.is_unique}"
)

print(
    f"Index range: "
    f"{biz_evaluation.index.min()} → "
    f"{biz_evaluation.index.max()}"
)

# ------------------------------------------------------------
# 5. Required business columns
# ------------------------------------------------------------

business_columns = [
    "Risk_Regime",
    "Portfolio_Exposure",
    "Static_Portfolio_Return",
    "Adaptive_Net_Return"
]

print("\n" + "=" * 80)
print("BUSINESS COLUMN AVAILABILITY")
print("=" * 80)

for column in business_columns:
    
    if column in biz_evaluation.columns:
        print(
            f"{column:<30} FOUND"
        )
    else:
        print(
            f"{column:<30} MISSING"
        )

# ------------------------------------------------------------
# 6. Risk regime distribution
# ------------------------------------------------------------

if "Risk_Regime" in biz_evaluation.columns:

    print("\n" + "=" * 80)
    print("RISK REGIME DISTRIBUTION")
    print("=" * 80)

    print(
        biz_evaluation["Risk_Regime"]
        .value_counts(dropna=False)
        .to_string()
    )

# ------------------------------------------------------------
# 7. Exposure distribution
# ------------------------------------------------------------

if "Portfolio_Exposure" in biz_evaluation.columns:

    print("\n" + "=" * 80)
    print("PORTFOLIO EXPOSURE DISTRIBUTION")
    print("=" * 80)

    print(
        biz_evaluation["Portfolio_Exposure"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

# ------------------------------------------------------------
# 8. Date sample
# ------------------------------------------------------------

if "Date" in biz_evaluation.columns:

    print("\n" + "=" * 80)
    print("FIRST 10 BUSINESS-EVALUATION DATES")
    print("=" * 80)

    print(
        biz_evaluation[
            ["Date"]
        ]
        .head(10)
        .to_string(index=False)
    )

    print("\n" + "=" * 80)
    print("LAST 10 BUSINESS-EVALUATION DATES")
    print("=" * 80)

    print(
        biz_evaluation[
            ["Date"]
        ]
        .tail(10)
        .to_string(index=False)
    )

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE — NO DATA MODIFIED")
print("=" * 80)

BUSINESS EVALUATION DATA LINEAGE DIAGNOSTIC

OBJECT EXISTENCE
--------------------------------------------------------------------------------
biz_evaluation            EXISTS | type=DataFrame       | shape=(178, 28)
evaluation                NOT FOUND
test_data_vix_timing      NOT FOUND
test_data_vix_valid       NOT FOUND
clean_nifty               NOT FOUND
oos_forecast_matrix       NOT FOUND
forecast_matrix           NOT FOUND

BIZ_EVALUATION STRUCTURE
Shape: (178, 28)

Columns:
  1. Date
  2. NIFTY_Log_Return
  3. Actual_Volatility
  4. India_VIX_PreReturn
  5. Historical_20D
  6. EWMA
  7. GARCH_t
  8. GJR_GARCH_t
  9. EGARCH_t
 10. Historical_20D_VIX
 11. EWMA_VIX
 12. EGARCH_t_VIX
 13. Forecasted_Volatility
 14. Risk_Regime
 15. Portfolio_Exposure
 16. NIFTY_Simple_Return
 17. Static_Exposure
 18. Static_Portfolio_Return
 19. Adaptive_Portfolio_Return
 20. Static_Wealth
 21. Adaptive_Wealth
 22. Exposure_Change
 23. Absolute_Exposure_Change
 24. Exposure_Changed
 25. Regime_Chang

Risk_Regime
Extreme     50
Normal      48
Elevated    45
Low         24
High        11

PORTFOLIO EXPOSURE DISTRIBUTION
Portfolio_Exposure
0.4    50
0.6    11
0.8    45
1.0    72

FIRST 10 BUSINESS-EVALUATION DATES
      Date
2026-01-01
2026-01-02
2026-01-05
2026-01-06
2026-01-07
2026-01-08
2026-01-09
2026-01-12
2026-01-13
2026-01-14

LAST 10 BUSINESS-EVALUATION DATES
      Date
2026-09-08
2026-09-09
2026-09-10
2026-09-11
2026-09-15
2026-09-16
2026-09-17
2026-09-18
2026-09-21
2026-09-22

DIAGNOSTIC COMPLETE — NO DATA MODIFIED


### Business Evaluation Final Integrity Check

This checkpoint confirms that the held-out 2026 business-evaluation dataset is complete, internally consistent, and ready for final business-impact interpretation.

The checkpoint does not recalculate forecasting models or modify the evaluation dataset. It verifies the evaluation period, observation count, required business outputs, portfolio-exposure bounds, return fields, and adaptive net-wealth series used by the downstream business-impact analysis.


In [30]:
# ============================================================
# FINAL INTEGRITY CHECK — BUSINESS EVALUATION
# ============================================================

print("=" * 80)
print("FINAL BUSINESS EVALUATION INTEGRITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Confirm business evaluation object
# ------------------------------------------------------------

assert "biz_evaluation" in globals(), (
    "biz_evaluation is not available in the current kernel."
)

assert isinstance(biz_evaluation, pd.DataFrame), (
    "biz_evaluation is not a pandas DataFrame."
)

# ------------------------------------------------------------
# 2. Expected evaluation period
# ------------------------------------------------------------

EXPECTED_START = pd.Timestamp("2026-01-01")
EXPECTED_END = pd.Timestamp("2026-09-22")
EXPECTED_OBSERVATIONS = 178

assert biz_evaluation["Date"].min() == EXPECTED_START, (
    f"Unexpected evaluation start: {biz_evaluation['Date'].min()}"
)

assert biz_evaluation["Date"].max() == EXPECTED_END, (
    f"Unexpected evaluation end: {biz_evaluation['Date'].max()}"
)

assert len(biz_evaluation) == EXPECTED_OBSERVATIONS, (
    f"Unexpected observation count: {len(biz_evaluation)}"
)

assert biz_evaluation["Date"].nunique() == EXPECTED_OBSERVATIONS, (
    "Duplicate evaluation dates detected."
)

# ------------------------------------------------------------
# 3. Required business columns
# ------------------------------------------------------------

required_business_columns = [
    "Date",
    "Forecasted_Volatility",
    "Risk_Regime",
    "Portfolio_Exposure",
    "NIFTY_Simple_Return",
    "Static_Portfolio_Return",
    "Adaptive_Portfolio_Return",
    "Static_Wealth",
    "Adaptive_Wealth",
    "Exposure_Change",
    "Absolute_Exposure_Change",
    "Daily_Transaction_Cost",
    "Adaptive_Net_Return",
    "Adaptive_Net_Wealth"
]

missing_columns = [
    column
    for column in required_business_columns
    if column not in biz_evaluation.columns
]

assert not missing_columns, (
    f"Missing required business columns: {missing_columns}"
)

# ------------------------------------------------------------
# 4. Missing-value check
# ------------------------------------------------------------

required_data = biz_evaluation[required_business_columns]

missing_values = required_data.isna().sum()

assert missing_values.sum() == 0, (
    "Missing values detected in required business-evaluation fields."
)

# ------------------------------------------------------------
# 5. Exposure integrity
# ------------------------------------------------------------

assert (
    biz_evaluation["Portfolio_Exposure"].between(0, 1).all()
), "Portfolio exposure contains values outside [0, 1]."

# ------------------------------------------------------------
# 6. Return and wealth integrity
# ------------------------------------------------------------

return_columns = [
    "NIFTY_Simple_Return",
    "Static_Portfolio_Return",
    "Adaptive_Portfolio_Return",
    "Daily_Transaction_Cost",
    "Adaptive_Net_Return"
]

for column in return_columns:
    assert np.isfinite(
        biz_evaluation[column].to_numpy()
    ).all(), f"Non-finite values detected in {column}."

wealth_columns = [
    "Static_Wealth",
    "Adaptive_Wealth",
    "Adaptive_Net_Wealth"
]

for column in wealth_columns:
    assert np.isfinite(
        biz_evaluation[column].to_numpy()
    ).all(), f"Non-finite values detected in {column}."

# ------------------------------------------------------------
# 7. Transaction-cost integrity
# ------------------------------------------------------------

assert (
    biz_evaluation["Daily_Transaction_Cost"] >= 0
).all(), "Negative transaction costs detected."

assert (
    biz_evaluation["Absolute_Exposure_Change"] >= 0
).all(), "Negative absolute exposure changes detected."

# ------------------------------------------------------------
# 8. Risk-regime integrity
# ------------------------------------------------------------

expected_regimes = {
    "Low",
    "Normal",
    "Elevated",
    "High",
    "Extreme"
}

observed_regimes = set(
    biz_evaluation["Risk_Regime"].dropna().unique()
)

assert observed_regimes.issubset(expected_regimes), (
    f"Unexpected risk regime(s): "
    f"{observed_regimes - expected_regimes}"
)

# ------------------------------------------------------------
# 9. Chronological integrity
# ------------------------------------------------------------

assert biz_evaluation["Date"].is_monotonic_increasing, (
    "Business evaluation dates are not chronological."
)

# ------------------------------------------------------------
# 10. Final diagnostic output
# ------------------------------------------------------------

print("\nEvaluation period:")
print(
    f"{biz_evaluation['Date'].min().date()} "
    f"to "
    f"{biz_evaluation['Date'].max().date()}"
)

print(f"Observations: {len(biz_evaluation)}")

print(
    f"Average portfolio exposure: "
    f"{biz_evaluation['Portfolio_Exposure'].mean():.2%}"
)

print(
    f"Exposure range: "
    f"{biz_evaluation['Portfolio_Exposure'].min():.0%} "
    f"to "
    f"{biz_evaluation['Portfolio_Exposure'].max():.0%}"
)

print(
    f"High + Extreme observations: "
    f"{biz_evaluation['Risk_Regime'].isin(['High', 'Extreme']).sum()}"
)

print(
    f"Total transaction cost: "
    f"{biz_evaluation['Daily_Transaction_Cost'].sum():.4%}"
)

print("\n" + "=" * 80)
print("FINAL BUSINESS EVALUATION INTEGRITY CHECK: PASS")
print("No data modified.")
print("Evaluation state is ready for final business-impact reporting.")
print("=" * 80)

FINAL BUSINESS EVALUATION INTEGRITY CHECK

Evaluation period:
2026-01-01 to 2026-09-22
Observations: 178
Average portfolio exposure: 75.62%
Exposure range: 40% to 100%
High + Extreme observations: 61
Total transaction cost: 0.6400%

FINAL BUSINESS EVALUATION INTEGRITY CHECK: PASS
No data modified.
Evaluation state is ready for final business-impact reporting.


In [31]:
# ============================================================
# FINAL BUSINESS IMPACT KPI SUMMARY — 2026 EVALUATION
# ============================================================

print("=" * 80)
print("FINAL BUSINESS IMPACT KPI SUMMARY — 2026 EVALUATION")
print("=" * 80)

print("\nPERFORMANCE")
print("-" * 80)
print(
    f"Static compounded return:        "
    f"{biz_evaluation['Static_Wealth'].iloc[-1] - 1:.2%}"
)
print(
    f"Adaptive net compounded return:  "
    f"{biz_adaptive_net_total_return:.2%}"
)
print(
    f"Total return advantage:           "
    f"{biz_net_return_advantage:.2%}"
)

print("\nRISK REDUCTION")
print("-" * 80)
print(
    f"Annualized volatility reduction: "
    f"{biz_net_volatility_reduction:.2%}"
)
print(
    f"Maximum drawdown reduction:       "
    f"{biz_net_drawdown_reduction:.2%}"
)

print("\nEXPOSURE MANAGEMENT")
print("-" * 80)
print(
    f"Average adaptive exposure:        "
    f"{biz_average_exposure:.2%}"
)
print(
    f"Average exposure reduction:       "
    f"{biz_exposure_reduction:.2%}"
)
print(
    f"High + Extreme observations:      "
    f"{biz_protection_observations}"
)
print(
    f"High + Extreme exposure:          "
    f"{biz_average_protection_exposure:.2%}"
)

print("\nTRANSACTION COST")
print("-" * 80)
print(
    f"Transaction-cost assumption:      "
    f"{biz_central_cost_rate:.2%}"
)
print(
    f"Total transaction cost:            "
    f"{biz_total_transaction_cost:.2%}"
)

print("\nRISK-ADJUSTED PERFORMANCE")
print("-" * 80)
print(
    f"Sharpe difference:                "
    f"{biz_net_sharpe_difference:.4f}"
)
print(
    f"Sortino difference:               "
    f"{biz_net_sortino_difference:.4f}"
)

print("\n" + "=" * 80)
print("BUSINESS INTERPRETATION")
print("=" * 80)

print(
    "The adaptive framework reduced portfolio exposure during "
    "higher-risk regimes and produced a smaller compounded loss "
    "than the static benchmark during the 2026 evaluation period."
)

print(
    "The principal business benefit was downside-risk control, "
    "reflected in lower volatility and maximum drawdown."
)

print(
    "Risk-adjusted performance did not improve: Sharpe and Sortino "
    "differences remained slightly negative."
)

print(
    "Therefore, the framework should be interpreted as a "
    "risk-management and exposure-control system rather than "
    "a guaranteed return-enhancement strategy."
)

print("\n" + "=" * 80)
print("FINAL BUSINESS IMPACT SUMMARY: VALID")
print("=" * 80)

FINAL BUSINESS IMPACT KPI SUMMARY — 2026 EVALUATION

PERFORMANCE
--------------------------------------------------------------------------------
Static compounded return:        -10.72%
Adaptive net compounded return:  -6.65%
Total return advantage:           4.06%

RISK REDUCTION
--------------------------------------------------------------------------------
Annualized volatility reduction: 38.12%
Maximum drawdown reduction:       43.10%

EXPOSURE MANAGEMENT
--------------------------------------------------------------------------------
Average adaptive exposure:        75.62%
Average exposure reduction:       24.38%
High + Extreme observations:      61
High + Extreme exposure:          43.61%

TRANSACTION COST
--------------------------------------------------------------------------------
Transaction-cost assumption:      0.10%
Total transaction cost:            0.64%

RISK-ADJUSTED PERFORMANCE
--------------------------------------------------------------------------------
Sharp

## Final Business Impact Methodology and Validation

### Purpose

This notebook translates the validated out-of-sample volatility forecasting results into a practical business-impact framework. The objective is not to demonstrate guaranteed profitability, but to evaluate whether forecasted market-risk conditions can support dynamic exposure management and reduce downside risk relative to a static full-exposure benchmark.

### Evaluation Design

The business evaluation uses the frozen Research Output v1 and evaluates the adaptive exposure framework exclusively on the held-out 2026 period from **1 January 2026 to 22 September 2026**, comprising **178 observations**.

The 2024–2025 period was used as the business-development period for selecting the primary forecasting signal and calibrating the exposure-policy framework. The 2026 period was then kept as the held-out business evaluation period.

This separation prevents the 2026 observations from influencing model selection or policy calibration.

### Primary Forecasting Signal

The primary forecasting model was selected using development-period performance only. Among the evaluated forecasting alternatives, **EGARCH_t_VIX** produced the lowest development-period RMSE and was therefore selected as the primary signal for the business-impact evaluation.

The selected signal was subsequently used to classify market-risk conditions into five regimes:

* Low
* Normal
* Elevated
* High
* Extreme

These regimes were mapped to predetermined portfolio-exposure levels ranging from **100% exposure in lower-risk conditions to 40% exposure in the Extreme regime**.

### Exposure-Management Framework

The business framework reduces exposure as forecasted volatility increases. The resulting policy is:

| Risk Regime | Portfolio Exposure |
| ----------- | -----------------: |
| Low         |               100% |
| Normal      |               100% |
| Elevated    |                80% |
| High        |                60% |
| Extreme     |                40% |

The policy is interpreted as a risk-budget and exposure-management mechanism rather than as a claim that the model predicts market direction.

### Performance Measurement

The analysis distinguishes between arithmetic daily-return diagnostics and actual compounded portfolio performance.

For portfolio-level performance, wealth is calculated using:

$$
W_t = \prod_{i=1}^{t}(1+r_i)
$$

and the compounded return is calculated as:

$$
R_{\text{compounded}} = W_T - 1
$$

Arithmetic sums of daily returns are retained only as supplementary diagnostics and are not interpreted as compounded portfolio returns.

### Transaction-Cost Sensitivity

The adaptive strategy was evaluated under illustrative transaction-cost assumptions of **5, 10, and 20 basis points** per unit of absolute exposure change.

The central business-impact scenario uses a **10-basis-point transaction-cost assumption**. Under this scenario, the total estimated transaction cost over the evaluation period was **0.64%**.

These transaction-cost results are sensitivity assumptions rather than observed execution costs.

### Final Out-of-Sample Business Results

During the 2026 held-out evaluation period:

* Static compounded return: **−10.72%**
* Adaptive net compounded return: **−6.65%**
* Compounded return advantage versus static exposure: **+4.06 percentage points**
* Annualized volatility reduction: **38.12%**
* Maximum drawdown reduction: **43.10%**
* Average adaptive exposure: **75.62%**
* Average exposure reduction: **24.38%**
* High + Extreme observations: **61 of 178 (34.27%)**
* Average exposure during High + Extreme regimes: **43.61%**
* Central transaction-cost assumption: **10 bps**
* Total estimated transaction cost: **0.64%**

The adaptive framework therefore produced a smaller compounded loss than the static benchmark while substantially reducing volatility and maximum drawdown.

### Risk-Adjusted Performance

Risk-adjusted performance was evaluated separately from absolute loss reduction.

The differences were:

* Sharpe difference: **−0.0082**
* Sortino difference: **−0.0413**

Therefore, the adaptive framework did **not** demonstrate an improvement in conventional risk-adjusted return measures during the held-out evaluation period.

This distinction is important: lower volatility and lower drawdown do not automatically imply superior Sharpe or Sor


Auditing

In [32]:
"""from pathlib import Path
import nbformat
import json

# ============================================================
# COMPLETE AUDIT EXPORT
# 03_BUSINESS_IMPACT NOTEBOOK
# SOURCE + ALL STORED OUTPUTS
# ============================================================

PROJECT_ROOT = Path(
    r"D:\DS Projects\Adaptive Market Risk Intelligence System"
)

NOTEBOOK_DIR = PROJECT_ROOT / "Notebooks"

# ============================================================
# SEPARATE AUDIT DIRECTORY
# ============================================================

AUDIT_DIR = PROJECT_ROOT / "Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SOURCE NOTEBOOK
# ============================================================

SOURCE_NOTEBOOK = "03_business_impact.ipynb"

source_notebook = NOTEBOOK_DIR / SOURCE_NOTEBOOK

# ============================================================
# OUTPUT FILE
# ============================================================

OUTPUT_FILE = (
    AUDIT_DIR
    / "Audit_03_business_impact_COMPLETE.txt"
)

# ============================================================
# VERIFY NOTEBOOK
# ============================================================

if not source_notebook.exists():
    raise FileNotFoundError(
        f"Notebook not found:\n{source_notebook}"
    )

# ============================================================
# READ SAVED NOTEBOOK
# ============================================================

with open(
    source_notebook,
    "r",
    encoding="utf-8"
) as f:

    nb = nbformat.read(
        f,
        as_version=4
    )

total_cells = len(nb.cells)

# ============================================================
# CREATE SINGLE AUDIT FILE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as out:

    # --------------------------------------------------------
    # FILE HEADER
    # --------------------------------------------------------

    out.write(
        "=" * 120 + "\n"
    )

    out.write(
        "ADAPTIVE MARKET RISK INTELLIGENCE SYSTEM\n"
    )

    out.write(
        "COMPLETE BUSINESS IMPACT NOTEBOOK AUDIT\n"
    )

    out.write(
        "=" * 120 + "\n\n"
    )

    out.write(
        f"Source notebook: {source_notebook}\n"
    )

    out.write(
        f"Audit output directory: {AUDIT_DIR}\n"
    )

    out.write(
        f"Total notebook cells: {total_cells}\n"
    )

    out.write(
        "Audited cells: 001–"
        f"{total_cells:03d}\n"
    )

    out.write(
        "\n"
    )

    # --------------------------------------------------------
    # EACH CELL
    # --------------------------------------------------------

    for cell_number in range(
        1,
        total_cells + 1
    ):

        cell = nb.cells[cell_number - 1]

        out.write(
            "\n\n"
            + "#" * 120
            + "\n"
        )

        out.write(
            f"CELL {cell_number:03d}\n"
        )

        out.write(
            "#" * 120
            + "\n\n"
        )

        # ====================================================
        # CELL METADATA
        # ====================================================

        out.write(
            "=" * 100 + "\n"
        )

        out.write(
            "CELL INFORMATION\n"
        )

        out.write(
            "=" * 100 + "\n\n"
        )

        out.write(
            f"Cell number: {cell_number}\n"
        )

        out.write(
            f"Cell type: {cell.cell_type}\n"
        )

        if cell.cell_type == "code":

            out.write(
                "Execution count: "
                f"{cell.get('execution_count', None)}\n"
            )

        out.write(
            "\n"
        )

        # ====================================================
        # COMPLETE SOURCE / CONTENT
        # ====================================================

        out.write(
            "=" * 100 + "\n"
        )

        out.write(
            "BEGIN COMPLETE SOURCE / CONTENT\n"
        )

        out.write(
            "=" * 100 + "\n\n"
        )

        out.write(
            cell.source
        )

        if not cell.source.endswith("\n"):
            out.write("\n")

        out.write(
            "\n"
            + "=" * 100
            + "\n"
        )

        out.write(
            "END COMPLETE SOURCE / CONTENT\n"
        )

        out.write(
            "=" * 100
            + "\n\n"
        )

        # ====================================================
        # NON-CODE CELL
        # ====================================================

        if cell.cell_type != "code":

            out.write(
                "[NON-CODE CELL — NO EXECUTION OUTPUT]\n"
            )

            continue

        # ====================================================
        # STORED OUTPUTS
        # ====================================================

        outputs = cell.get(
            "outputs",
            []
        )

        out.write(
            "=" * 100 + "\n"
        )

        out.write(
            f"BEGIN STORED OUTPUTS "
            f"({len(outputs)} OUTPUT OBJECTS)\n"
        )

        out.write(
            "=" * 100 + "\n\n"
        )

        if not outputs:

            out.write(
                "[NO STORED OUTPUT]\n"
            )

        # ====================================================
        # EACH OUTPUT OBJECT
        # ====================================================

        for output_number, output in enumerate(
            outputs,
            start=1
        ):

            output_type = output.get(
                "output_type",
                "unknown"
            )

            out.write(
                "\n"
                + "-" * 90
                + "\n"
            )

            out.write(
                f"OUTPUT {output_number}\n"
            )

            out.write(
                f"OUTPUT TYPE: {output_type}\n"
            )

            out.write(
                "-" * 90
                + "\n\n"
            )

            # ------------------------------------------------
            # STREAM OUTPUT
            # ------------------------------------------------

            if output_type == "stream":

                value = output.get(
                    "text",
                    ""
                )

                if isinstance(value, list):

                    value = "".join(value)

                out.write(
                    "STREAM CONTENT:\n\n"
                )

                out.write(
                    str(value)
                )

                if not str(value).endswith("\n"):

                    out.write("\n")

            # ------------------------------------------------
            # ERROR OUTPUT
            # ------------------------------------------------

            elif output_type == "error":

                out.write(
                    "EXCEPTION NAME:\n"
                )

                out.write(
                    str(
                        output.get(
                            "ename",
                            ""
                        )
                    )
                )

                out.write(
                    "\n\n"
                )

                out.write(
                    "EXCEPTION MESSAGE:\n"
                )

                out.write(
                    str(
                        output.get(
                            "evalue",
                            ""
                        )
                    )
                )

                out.write(
                    "\n\n"
                )

                out.write(
                    "COMPLETE TRACEBACK:\n\n"
                )

                traceback = output.get(
                    "traceback",
                    []
                )

                if isinstance(
                    traceback,
                    list
                ):

                    for line in traceback:

                        out.write(
                            str(line)
                        )

                        if not str(
                            line
                        ).endswith("\n"):

                            out.write("\n")

                else:

                    out.write(
                        str(traceback)
                    )

                    out.write(
                        "\n"
                    )

            # ------------------------------------------------
            # DISPLAY / EXECUTE RESULT
            # ------------------------------------------------

            elif output_type in (
                "display_data",
                "execute_result"
            ):

                data = output.get(
                    "data",
                    {}
                )

                if not data:

                    out.write(
                        "[OUTPUT HAS NO DATA FIELD]\n"
                    )

                for mime_type, value in data.items():

                    out.write(
                        "\n"
                        + "MIME TYPE: "
                        + mime_type
                        + "\n"
                    )

                    out.write(
                        "-" * 70
                        + "\n"
                    )

                    if isinstance(
                        value,
                        str
                    ):

                        out.write(
                            value
                        )

                        if not value.endswith("\n"):

                            out.write(
                                "\n"
                            )

                    else:

                        try:

                            serialized = json.dumps(
                                value,
                                ensure_ascii=False,
                                indent=2
                            )

                        except Exception:

                            serialized = str(
                                value
                            )

                        out.write(
                            serialized
                        )

                        out.write(
                            "\n"
                        )

                    if mime_type in (
                        "image/png",
                        "image/jpeg",
                        "image/svg+xml"
                    ):

                        out.write(
                            "\n"
                            "[IMAGE DATA IS STORED "
                            "IN THE NOTEBOOK]\n"
                        )

            # ------------------------------------------------
            # UNKNOWN OUTPUT
            # ------------------------------------------------

            else:

                out.write(
                    "RAW OUTPUT OBJECT:\n\n"
                )

                try:

                    raw = json.dumps(
                        output,
                        ensure_ascii=False,
                        indent=2
                    )

                except Exception:

                    raw = str(
                        output
                    )

                out.write(
                    raw
                )

                out.write(
                    "\n"
                )

        # ====================================================
        # END OUTPUTS
        # ====================================================

        out.write(
            "\n"
            + "=" * 100
            + "\n"
        )

        out.write(
            "END STORED OUTPUTS\n"
        )

        out.write(
            "=" * 100
            + "\n"
        )

    # ========================================================
    # FINAL FILE MARKER
    # ========================================================

    out.write(
        "\n\n"
        + "=" * 120
        + "\n"
    )

    out.write(
        "END COMPLETE BUSINESS IMPACT NOTEBOOK AUDIT\n"
    )

    out.write(
        "=" * 120
        + "\n"
    )

# ============================================================
# VERIFICATION
# ============================================================

print(
    "=" * 80
)

print(
    "BUSINESS IMPACT AUDIT EXPORT COMPLETE"
)

print(
    "=" * 80
)

print(
    f"\nSource notebook:"
)

print(
    source_notebook
)

print(
    f"\nTotal cells exported:"
)

print(
    total_cells
)

print(
    "\nAudit directory:"
)

print(
    AUDIT_DIR
)

print(
    "\nOutput file:"
)

print(
    OUTPUT_FILE
)

print(
    f"\nFile size:"
)

print(
    f"{OUTPUT_FILE.stat().st_size:,} bytes"
)

print(
    "\n✓ Complete cell source/content exported"
)

print(
    "✓ Stored outputs exported"
)

print(
    "✓ Errors and tracebacks exported"
)

print(
    "✓ Cell order preserved"
)

print(
    "✓ Entire 03_business_impact notebook audited"
)

print(
    "✓ Audit file stored OUTSIDE the Notebooks folder"
)

print(
    "\nAudit file:"
)

print(
    OUTPUT_FILE.name
)
"""

<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:12: SyntaxWarning: invalid escape sequence '\D'
C:\Users\user\AppData\Local\Temp\ipykernel_13736\1263074166.py:12: SyntaxWarning: invalid escape sequence '\D'
  r"D:\DS Projects\Adaptive Market Risk Intelligence System"


'from pathlib import Path\nimport nbformat\nimport json\n\n# ============================================================\n# COMPLETE AUDIT EXPORT\n# 03_BUSINESS_IMPACT NOTEBOOK\n# SOURCE + ALL STORED OUTPUTS\n# ============================================================\n\nPROJECT_ROOT = Path(\n    r"D:\\DS Projects\\Adaptive Market Risk Intelligence System"\n)\n\nNOTEBOOK_DIR = PROJECT_ROOT / "Notebooks"\n\n# ============================================================\n# SEPARATE AUDIT DIRECTORY\n# ============================================================\n\nAUDIT_DIR = PROJECT_ROOT / "Audit"\nAUDIT_DIR.mkdir(parents=True, exist_ok=True)\n\n# ============================================================\n# SOURCE NOTEBOOK\n# ============================================================\n\nSOURCE_NOTEBOOK = "03_business_impact.ipynb"\n\nsource_notebook = NOTEBOOK_DIR / SOURCE_NOTEBOOK\n\n# ============================================================\n# OUTPUT FILE\n# ==============